# Queries Avançadas: CTEs, Window Functions e LATERAL JOIN

Este notebook cobre as ferramentas de query que separam SQL básico de SQL de produção: CTEs para compor lógica sem aninhar subqueries, window functions para calcular agregados mantendo granularidade de linha, LATERAL JOIN para o padrão "top-N por grupo", e agregações condicionais para montar pivots sem CROSSTAB. Cada técnica é demonstrada com queries executáveis contra um schema de crédito (clientes, propostas, parcelas) e o notebook fecha com um dashboard gerencial que combina tudo.

**Pré-requisitos:**
- Conhecimento de SELECT, JOIN e GROUP BY
- Notebook 01 (Fundamentos) e 02 (Índices e Performance) concluídos
- PostgreSQL 14+

**Schema usado:** sistema de crédito com propostas, clientes, parcelas, consultas externas e log de decisões.

## Como Window Functions Processam os Dados

![Fluxo de processamento de Window Functions](assets/03-queries-avancadas.png)

Window Functions executam **após** WHERE/FROM/JOIN mas **antes** do ORDER BY final. A cláusula `OVER(...)` define:
- `PARTITION BY`: divide em grupos (como GROUP BY, mas sem colapsar linhas)
- `ORDER BY`: ordena dentro de cada grupo
- `ROWS/RANGE BETWEEN`: define o frame (quais linhas a função "enxerga")

Diferença fundamental: GROUP BY colapsa N linhas em 1. OVER mantém todas as linhas e adiciona o resultado como coluna extra.

## Configuração da Conexão

Schema de crédito usado nos exemplos:
- `clientes`: cadastro com CPF, nome e renda mensal
- `propostas`: pedidos de crédito com valor, status e score
- `parcelas`: cronograma de pagamento por proposta
- `consultas_externas`: chamadas a bureaus de crédito (resposta em JSONB)
- `decisoes_log`: auditoria de cada etapa da esteira de crédito

In [1]:
from datetime import date, timedelta

import psycopg2
import psycopg2.extras

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "fintech_kb",
    "user": "postgres",
    "password": "postgres",
}

conn = psycopg2.connect(
    **DB_CONFIG,
    cursor_factory=psycopg2.extras.RealDictCursor,
)
conn.autocommit = True

print('Conexão estabelecida com sucesso.')

Conexão estabelecida com sucesso.


## Criação do Schema de Exemplo

DDL idempotente: `DROP TABLE IF EXISTS ... CASCADE` garante que rodar o notebook várias vezes não quebra.

In [2]:
DDL = """
DROP TABLE IF EXISTS parcelas, decisoes_log, consultas_externas, propostas, clientes CASCADE;

CREATE TABLE clientes (
    id          SERIAL PRIMARY KEY,
    cpf         CHAR(11)        NOT NULL UNIQUE,
    nome        VARCHAR(120)    NOT NULL,
    renda_mensal NUMERIC(12, 2) NOT NULL,
    created_at  TIMESTAMPTZ     NOT NULL DEFAULT now()
);

CREATE TABLE propostas (
    id          SERIAL PRIMARY KEY,
    numero      VARCHAR(20)     NOT NULL UNIQUE,
    cliente_id  INT             NOT NULL REFERENCES clientes(id),
    valor       NUMERIC(12, 2)  NOT NULL,
    status      VARCHAR(20)     NOT NULL CHECK (status IN ('pendente','aprovada','reprovada','cancelada')),
    score       SMALLINT        NOT NULL CHECK (score BETWEEN 0 AND 1000),
    created_at  TIMESTAMPTZ     NOT NULL DEFAULT now()
);

CREATE TABLE consultas_externas (
    id           SERIAL PRIMARY KEY,
    proposta_id  INT             NOT NULL REFERENCES propostas(id),
    provider     VARCHAR(40)     NOT NULL,
    response_data JSONB          NOT NULL DEFAULT '{}',
    status       VARCHAR(20)     NOT NULL,
    created_at   TIMESTAMPTZ     NOT NULL DEFAULT now()
);

CREATE TABLE decisoes_log (
    id              SERIAL PRIMARY KEY,
    proposta_id     INT             NOT NULL REFERENCES propostas(id),
    etapa           VARCHAR(40)     NOT NULL,
    resultado       VARCHAR(20)     NOT NULL,
    regras_aplicadas JSONB          NOT NULL DEFAULT '[]',
    executed_at     TIMESTAMPTZ     NOT NULL DEFAULT now()
);

CREATE TABLE parcelas (
    id              SERIAL PRIMARY KEY,
    proposta_id     INT             NOT NULL REFERENCES propostas(id),
    numero          SMALLINT        NOT NULL,
    valor           NUMERIC(10, 2)  NOT NULL,
    vencimento      DATE            NOT NULL,
    status_pagamento VARCHAR(20)    NOT NULL CHECK (status_pagamento IN ('aberta','paga','atrasada')),
    pago_em         DATE
);
"""

with conn.cursor() as cur:
    cur.execute(DDL)

print('Schema criado.')

Schema criado.


In [3]:
SEED = """
INSERT INTO clientes (cpf, nome, renda_mensal, created_at) VALUES
  ('11122233344', 'Ana Lima',       3500.00, now() - interval '180 days'),
  ('22233344455', 'Bruno Souza',    8200.00, now() - interval '160 days'),
  ('33344455566', 'Carla Mendes',  15000.00, now() - interval '140 days'),
  ('44455566677', 'Diego Rocha',    2800.00, now() - interval '120 days'),
  ('55566677788', 'Eva Santos',    22000.00, now() - interval '100 days'),
  ('66677788899', 'Fabio Nunes',    5100.00, now() - interval '90 days'),
  ('77788899900', 'Gabriela Cruz',  9700.00, now() - interval '80 days'),
  ('88899900011', 'Henrique Paz',   4300.00, now() - interval '70 days'),
  ('99900011122', 'Isabela Teles', 31000.00, now() - interval '60 days'),
  ('10011122233', 'Jonas Brito',    6600.00, now() - interval '50 days');

INSERT INTO propostas (numero, cliente_id, valor, status, score, created_at) VALUES
  ('PROP-001', 1,  5000.00, 'aprovada',   720, now() - interval '170 days'),
  ('PROP-002', 1, 12000.00, 'reprovada',  620, now() - interval '120 days'),
  ('PROP-003', 2, 30000.00, 'aprovada',   810, now() - interval '155 days'),
  ('PROP-004', 3, 80000.00, 'aprovada',   900, now() - interval '130 days'),
  ('PROP-005', 4,  3000.00, 'reprovada',  410, now() - interval '110 days'),
  ('PROP-006', 5, 50000.00, 'aprovada',   870, now() - interval '90 days'),
  ('PROP-007', 6,  8000.00, 'pendente',   680, now() - interval '20 days'),
  ('PROP-008', 7, 25000.00, 'aprovada',   790, now() - interval '75 days'),
  ('PROP-009', 8,  4500.00, 'cancelada',  530, now() - interval '65 days'),
  ('PROP-010', 9, 100000.00,'aprovada',   950, now() - interval '55 days'),
  ('PROP-011', 10, 15000.00,'aprovada',   740, now() - interval '45 days'),
  ('PROP-012', 1,  7000.00, 'aprovada',   730, now() - interval '30 days'),
  ('PROP-013', 2, 20000.00, 'pendente',   820, now() - interval '10 days'),
  ('PROP-014', 3, 60000.00, 'aprovada',   880, now() - interval '5 days');

INSERT INTO parcelas (proposta_id, numero, valor, vencimento, status_pagamento, pago_em)
SELECT
    p.id,
    gs.n,
    ROUND(p.valor / 12, 2),
    (p.created_at::date + (gs.n * 30)),
    CASE
        WHEN (p.created_at::date + (gs.n * 30)) < CURRENT_DATE - 5 THEN 'paga'
        WHEN (p.created_at::date + (gs.n * 30)) < CURRENT_DATE     THEN 'atrasada'
        ELSE 'aberta'
    END,
    CASE
        WHEN (p.created_at::date + (gs.n * 30)) < CURRENT_DATE - 5
        THEN (p.created_at::date + (gs.n * 30)) + 3
        ELSE NULL
    END
FROM propostas p
CROSS JOIN generate_series(1, 12) AS gs(n)
WHERE p.status = 'aprovada';
"""

with conn.cursor() as cur:
    cur.execute(SEED)

print('Dados de exemplo inseridos.')

Dados de exemplo inseridos.


## Função Auxiliar

In [4]:
def run(sql: str, params=None, limit: int = 20) -> list:
    """Executa SQL e retorna lista de dicts, imprimindo as primeiras `limit` linhas."""
    with conn.cursor() as cur:
        cur.execute(sql, params)
        rows = cur.fetchall()

    rows = [dict(r) for r in rows]
    for i, row in enumerate(rows[:limit]):
        print(row)
    if len(rows) > limit:
        print(f'... (+{len(rows) - limit} linhas)')
    return rows

print('run() definida.')

run() definida.


## 1. CTEs (WITH) — Queries Legíveis e Composíveis

Uma CTE (Common Table Expression) é um resultado temporário nomeado definido dentro de `WITH`.
Ela não persiste em disco: o otimizador do PostgreSQL decide se materializa ou faz inline.

**Quando usar CTE em vez de subquery:**
- A mesma lógica intermediária é referenciada mais de uma vez
- A query tem mais de dois níveis de aninhamento (legibilidade)
- O resultado intermediário precisa ser materializado para isolar efeitos de plano

**Sintaxe básica:**
```sql
WITH nome_cte AS (
    SELECT ...
)
SELECT * FROM nome_cte;
```

### 1.1 CTE Simples Substituindo Subquery

Sem CTE, a lógica de filtrar propostas aprovadas fica aninhada e é repetida se necessária em outro ponto.
Com CTE, o bloco `aprovadas` é definido uma vez e reutilizado de forma clara.

In [5]:
# Versão com subquery aninhada
sql_subquery = """
SELECT cliente_id, COUNT(*) AS total, SUM(valor) AS volume
FROM (
    SELECT cliente_id, valor
    FROM propostas
    WHERE status = 'aprovada'
) sub
GROUP BY cliente_id
ORDER BY volume DESC;
"""

# Versão equivalente com CTE - mais legível
sql_cte = """
WITH aprovadas AS (
    SELECT cliente_id, valor
    FROM propostas
    WHERE status = 'aprovada'
)
SELECT cliente_id, COUNT(*) AS total, SUM(valor) AS volume
FROM aprovadas
GROUP BY cliente_id
ORDER BY volume DESC;
"""

print('Volume aprovado por cliente (usando CTE):')
run(sql_cte)

Volume aprovado por cliente (usando CTE):
{'cliente_id': 3, 'total': 2, 'volume': Decimal('140000.00')}
{'cliente_id': 9, 'total': 1, 'volume': Decimal('100000.00')}
{'cliente_id': 5, 'total': 1, 'volume': Decimal('50000.00')}
{'cliente_id': 2, 'total': 1, 'volume': Decimal('30000.00')}
{'cliente_id': 7, 'total': 1, 'volume': Decimal('25000.00')}
{'cliente_id': 10, 'total': 1, 'volume': Decimal('15000.00')}
{'cliente_id': 1, 'total': 2, 'volume': Decimal('12000.00')}


[{'cliente_id': 3, 'total': 2, 'volume': Decimal('140000.00')},
 {'cliente_id': 9, 'total': 1, 'volume': Decimal('100000.00')},
 {'cliente_id': 5, 'total': 1, 'volume': Decimal('50000.00')},
 {'cliente_id': 2, 'total': 1, 'volume': Decimal('30000.00')},
 {'cliente_id': 7, 'total': 1, 'volume': Decimal('25000.00')},
 {'cliente_id': 10, 'total': 1, 'volume': Decimal('15000.00')},
 {'cliente_id': 1, 'total': 2, 'volume': Decimal('12000.00')}]

### 1.2 Múltiplas CTEs — Compondo um Relatório

CTEs podem ser encadeadas: cada uma pode referenciar as anteriores.
O exemplo abaixo monta um resumo por cliente em três etapas lógicas independentes,
que depois são combinadas no SELECT final.

In [6]:
sql_multi_cte = """
WITH
-- Etapa 1: resumo de propostas por cliente
resumo_propostas AS (
    SELECT
        cliente_id,
        COUNT(*)                                           AS total_propostas,
        COUNT(*) FILTER (WHERE status = 'aprovada')        AS aprovadas,
        COUNT(*) FILTER (WHERE status = 'reprovada')       AS reprovadas,
        SUM(valor) FILTER (WHERE status = 'aprovada')      AS volume_aprovado
    FROM propostas
    GROUP BY cliente_id
),
-- Etapa 2: taxa de aprovação
taxa AS (
    SELECT
        cliente_id,
        total_propostas,
        aprovadas,
        ROUND(aprovadas::numeric / NULLIF(total_propostas, 0) * 100, 1) AS taxa_aprovacao_pct,
        COALESCE(volume_aprovado, 0)                                     AS volume_aprovado
    FROM resumo_propostas
),
-- Etapa 3: join com dados do cliente
relatorio_final AS (
    SELECT
        c.nome,
        c.renda_mensal,
        t.total_propostas,
        t.aprovadas,
        t.taxa_aprovacao_pct,
        t.volume_aprovado
    FROM taxa t
    JOIN clientes c ON c.id = t.cliente_id
)
SELECT *
FROM relatorio_final
ORDER BY volume_aprovado DESC;
"""

print('Relatório completo por cliente:')
run(sql_multi_cte)

Relatório completo por cliente:
{'nome': 'Carla Mendes', 'renda_mensal': Decimal('15000.00'), 'total_propostas': 2, 'aprovadas': 2, 'taxa_aprovacao_pct': Decimal('100.0'), 'volume_aprovado': Decimal('140000.00')}
{'nome': 'Isabela Teles', 'renda_mensal': Decimal('31000.00'), 'total_propostas': 1, 'aprovadas': 1, 'taxa_aprovacao_pct': Decimal('100.0'), 'volume_aprovado': Decimal('100000.00')}
{'nome': 'Eva Santos', 'renda_mensal': Decimal('22000.00'), 'total_propostas': 1, 'aprovadas': 1, 'taxa_aprovacao_pct': Decimal('100.0'), 'volume_aprovado': Decimal('50000.00')}
{'nome': 'Bruno Souza', 'renda_mensal': Decimal('8200.00'), 'total_propostas': 2, 'aprovadas': 1, 'taxa_aprovacao_pct': Decimal('50.0'), 'volume_aprovado': Decimal('30000.00')}
{'nome': 'Gabriela Cruz', 'renda_mensal': Decimal('9700.00'), 'total_propostas': 1, 'aprovadas': 1, 'taxa_aprovacao_pct': Decimal('100.0'), 'volume_aprovado': Decimal('25000.00')}
{'nome': 'Jonas Brito', 'renda_mensal': Decimal('6600.00'), 'total_pro

[{'nome': 'Carla Mendes',
  'renda_mensal': Decimal('15000.00'),
  'total_propostas': 2,
  'aprovadas': 2,
  'taxa_aprovacao_pct': Decimal('100.0'),
  'volume_aprovado': Decimal('140000.00')},
 {'nome': 'Isabela Teles',
  'renda_mensal': Decimal('31000.00'),
  'total_propostas': 1,
  'aprovadas': 1,
  'taxa_aprovacao_pct': Decimal('100.0'),
  'volume_aprovado': Decimal('100000.00')},
 {'nome': 'Eva Santos',
  'renda_mensal': Decimal('22000.00'),
  'total_propostas': 1,
  'aprovadas': 1,
  'taxa_aprovacao_pct': Decimal('100.0'),
  'volume_aprovado': Decimal('50000.00')},
 {'nome': 'Bruno Souza',
  'renda_mensal': Decimal('8200.00'),
  'total_propostas': 2,
  'aprovadas': 1,
  'taxa_aprovacao_pct': Decimal('50.0'),
  'volume_aprovado': Decimal('30000.00')},
 {'nome': 'Gabriela Cruz',
  'renda_mensal': Decimal('9700.00'),
  'total_propostas': 1,
  'aprovadas': 1,
  'taxa_aprovacao_pct': Decimal('100.0'),
  'volume_aprovado': Decimal('25000.00')},
 {'nome': 'Jonas Brito',
  'renda_mensal':

### 1.3 CTE Recursiva — Dados Hierárquicos (Bônus)

A CTE recursiva é a ferramenta padrão do PostgreSQL para percorrer hierarquias
(organogramas, árvores de categorias, grafos acíclicos).

**Estrutura:**
```sql
WITH RECURSIVE nome AS (
    -- Anchor: ponto de partida (raiz)
    SELECT ...
    UNION ALL
    -- Parte recursiva: junta com o resultado anterior
    SELECT ... FROM tabela JOIN nome ON ...
)
SELECT * FROM nome;
```

O exemplo simula uma hierarquia de aprovação de crédito:

In [7]:
# Criar tabela hierárquica de exemplo
HIERARQUIA_DDL = """
DROP TABLE IF EXISTS etapas_fluxo;
CREATE TABLE etapas_fluxo (
    id        INT PRIMARY KEY,
    nome      VARCHAR(60) NOT NULL,
    pai_id    INT REFERENCES etapas_fluxo(id)
);
INSERT INTO etapas_fluxo VALUES
  (1, 'Análise de Crédito',   NULL),
  (2, 'Score Bureau',         1),
  (3, 'Renda Comprovada',     1),
  (4, 'Score Interno',        2),
  (5, 'Negativações',         2),
  (6, 'Holerite',             3),
  (7, 'Extrato Bancário',     3),
  (8, 'Decisão Final',        NULL),
  (9, 'Aprovação Automática', 8),
  (10,'Aprovação Manual',     8);
"""
with conn.cursor() as cur:
    cur.execute(HIERARQUIA_DDL)

sql_recursive = """
WITH RECURSIVE arvore AS (
    -- Anchor: raízes (sem pai)
    -- Cast para varchar sem limite para permitir concatenação na parte recursiva
    SELECT id, nome, pai_id, 0 AS nivel, nome::varchar AS caminho
    FROM etapas_fluxo
    WHERE pai_id IS NULL

    UNION ALL

    -- Parte recursiva: filhos das linhas já encontradas
    SELECT
        ef.id,
        ef.nome,
        ef.pai_id,
        a.nivel + 1,
        a.caminho || ' > ' || ef.nome
    FROM etapas_fluxo ef
    JOIN arvore a ON a.id = ef.pai_id
)
SELECT
    repeat('  ', nivel) || nome AS etapa_identada,
    nivel,
    caminho
FROM arvore
ORDER BY caminho;
"""

print('Hierarquia de etapas do fluxo de crédito:')
run(sql_recursive)

Hierarquia de etapas do fluxo de crédito:
{'etapa_identada': 'Análise de Crédito', 'nivel': 0, 'caminho': 'Análise de Crédito'}
{'etapa_identada': '  Renda Comprovada', 'nivel': 1, 'caminho': 'Análise de Crédito > Renda Comprovada'}
{'etapa_identada': '    Extrato Bancário', 'nivel': 2, 'caminho': 'Análise de Crédito > Renda Comprovada > Extrato Bancário'}
{'etapa_identada': '    Holerite', 'nivel': 2, 'caminho': 'Análise de Crédito > Renda Comprovada > Holerite'}
{'etapa_identada': '  Score Bureau', 'nivel': 1, 'caminho': 'Análise de Crédito > Score Bureau'}
{'etapa_identada': '    Negativações', 'nivel': 2, 'caminho': 'Análise de Crédito > Score Bureau > Negativações'}
{'etapa_identada': '    Score Interno', 'nivel': 2, 'caminho': 'Análise de Crédito > Score Bureau > Score Interno'}
{'etapa_identada': 'Decisão Final', 'nivel': 0, 'caminho': 'Decisão Final'}
{'etapa_identada': '  Aprovação Automática', 'nivel': 1, 'caminho': 'Decisão Final > Aprovação Automática'}
{'etapa_identada': '

[{'etapa_identada': 'Análise de Crédito',
  'nivel': 0,
  'caminho': 'Análise de Crédito'},
 {'etapa_identada': '  Renda Comprovada',
  'nivel': 1,
  'caminho': 'Análise de Crédito > Renda Comprovada'},
 {'etapa_identada': '    Extrato Bancário',
  'nivel': 2,
  'caminho': 'Análise de Crédito > Renda Comprovada > Extrato Bancário'},
 {'etapa_identada': '    Holerite',
  'nivel': 2,
  'caminho': 'Análise de Crédito > Renda Comprovada > Holerite'},
 {'etapa_identada': '  Score Bureau',
  'nivel': 1,
  'caminho': 'Análise de Crédito > Score Bureau'},
 {'etapa_identada': '    Negativações',
  'nivel': 2,
  'caminho': 'Análise de Crédito > Score Bureau > Negativações'},
 {'etapa_identada': '    Score Interno',
  'nivel': 2,
  'caminho': 'Análise de Crédito > Score Bureau > Score Interno'},
 {'etapa_identada': 'Decisão Final', 'nivel': 0, 'caminho': 'Decisão Final'},
 {'etapa_identada': '  Aprovação Automática',
  'nivel': 1,
  'caminho': 'Decisão Final > Aprovação Automática'},
 {'etapa_ide

## 2. Window Functions — Análise sem Perder Linhas

Window Functions diferem de GROUP BY: elas calculam agregados mas **mantêm todas as linhas** do resultado.
A cláusula `OVER (...)` define a janela (partição + ordenação + frame).

**Estrutura geral:**
```sql
funcao() OVER (
    PARTITION BY coluna_agrupamento   -- divide em grupos (opcional)
    ORDER BY coluna_ordenacao         -- define ordem dentro do grupo
    ROWS BETWEEN ... AND ...          -- frame (opcional)
)
```

**Quando Window Functions executam:** após WHERE e JOIN, mas antes de HAVING e ORDER BY final.
Por isso, não é possível filtrar pelo resultado de uma window function sem usar uma subquery ou CTE.

### 2.1 ROW_NUMBER — Ranking de Clientes por Score dentro de Faixa de Renda

`ROW_NUMBER()` atribui um número sequencial único dentro de cada partição.
Útil para pegar o "primeiro de cada grupo" (top-N por grupo).

In [8]:
sql_row_number = """
SELECT
    c.nome,
    c.renda_mensal,
    p.score,
    -- Classifica faixa de renda para particionar
    CASE
        WHEN c.renda_mensal < 5000  THEN 'ate_5k'
        WHEN c.renda_mensal < 15000 THEN '5k_15k'
        ELSE 'acima_15k'
    END AS faixa_renda,
    -- Ranking do cliente dentro da faixa, pelo maior score
    ROW_NUMBER() OVER (
        PARTITION BY
            CASE
                WHEN c.renda_mensal < 5000  THEN 'ate_5k'
                WHEN c.renda_mensal < 15000 THEN '5k_15k'
                ELSE 'acima_15k'
            END
        ORDER BY p.score DESC
    ) AS rank_na_faixa
FROM propostas p
JOIN clientes c ON c.id = p.cliente_id
WHERE p.status = 'aprovada'
ORDER BY faixa_renda, rank_na_faixa;
"""

print('Ranking de score por faixa de renda:')
run(sql_row_number)

Ranking de score por faixa de renda:
{'nome': 'Bruno Souza', 'renda_mensal': Decimal('8200.00'), 'score': 810, 'faixa_renda': '5k_15k', 'rank_na_faixa': 1}
{'nome': 'Gabriela Cruz', 'renda_mensal': Decimal('9700.00'), 'score': 790, 'faixa_renda': '5k_15k', 'rank_na_faixa': 2}
{'nome': 'Jonas Brito', 'renda_mensal': Decimal('6600.00'), 'score': 740, 'faixa_renda': '5k_15k', 'rank_na_faixa': 3}
{'nome': 'Isabela Teles', 'renda_mensal': Decimal('31000.00'), 'score': 950, 'faixa_renda': 'acima_15k', 'rank_na_faixa': 1}
{'nome': 'Carla Mendes', 'renda_mensal': Decimal('15000.00'), 'score': 900, 'faixa_renda': 'acima_15k', 'rank_na_faixa': 2}
{'nome': 'Carla Mendes', 'renda_mensal': Decimal('15000.00'), 'score': 880, 'faixa_renda': 'acima_15k', 'rank_na_faixa': 3}
{'nome': 'Eva Santos', 'renda_mensal': Decimal('22000.00'), 'score': 870, 'faixa_renda': 'acima_15k', 'rank_na_faixa': 4}
{'nome': 'Ana Lima', 'renda_mensal': Decimal('3500.00'), 'score': 730, 'faixa_renda': 'ate_5k', 'rank_na_faix

[{'nome': 'Bruno Souza',
  'renda_mensal': Decimal('8200.00'),
  'score': 810,
  'faixa_renda': '5k_15k',
  'rank_na_faixa': 1},
 {'nome': 'Gabriela Cruz',
  'renda_mensal': Decimal('9700.00'),
  'score': 790,
  'faixa_renda': '5k_15k',
  'rank_na_faixa': 2},
 {'nome': 'Jonas Brito',
  'renda_mensal': Decimal('6600.00'),
  'score': 740,
  'faixa_renda': '5k_15k',
  'rank_na_faixa': 3},
 {'nome': 'Isabela Teles',
  'renda_mensal': Decimal('31000.00'),
  'score': 950,
  'faixa_renda': 'acima_15k',
  'rank_na_faixa': 1},
 {'nome': 'Carla Mendes',
  'renda_mensal': Decimal('15000.00'),
  'score': 900,
  'faixa_renda': 'acima_15k',
  'rank_na_faixa': 2},
 {'nome': 'Carla Mendes',
  'renda_mensal': Decimal('15000.00'),
  'score': 880,
  'faixa_renda': 'acima_15k',
  'rank_na_faixa': 3},
 {'nome': 'Eva Santos',
  'renda_mensal': Decimal('22000.00'),
  'score': 870,
  'faixa_renda': 'acima_15k',
  'rank_na_faixa': 4},
 {'nome': 'Ana Lima',
  'renda_mensal': Decimal('3500.00'),
  'score': 730,


### 2.2 RANK vs DENSE_RANK — O que muda com empates

Quando duas linhas têm o mesmo valor ordenado:
- `RANK()`: ambas recebem o mesmo número e o próximo é pulado (1, 2, 2, **4**)
- `DENSE_RANK()`: ambas recebem o mesmo número mas o próximo continua em sequência (1, 2, 2, **3**)
- `ROW_NUMBER()`: nunca empata — o desempate é arbitrário se não houver ORDER BY determinístico

In [9]:
# Insere duas propostas com o mesmo score para demonstrar o empate
# Score 900 para que fiquem no top 8 e o empate seja visível
with conn.cursor() as cur:
    cur.execute("""
        INSERT INTO propostas (numero, cliente_id, valor, status, score)
        VALUES ('PROP-EMPATE-A', 4, 5000, 'aprovada', 900),
               ('PROP-EMPATE-B', 6, 5000, 'aprovada', 900);
    """)

sql_rank_diff = """
SELECT
    numero,
    score,
    ROW_NUMBER()  OVER (ORDER BY score DESC) AS row_number,
    RANK()        OVER (ORDER BY score DESC) AS rank,
    DENSE_RANK()  OVER (ORDER BY score DESC) AS dense_rank
FROM propostas
WHERE status = 'aprovada'
ORDER BY score DESC
LIMIT 10;
"""

print('Comparação entre ROW_NUMBER, RANK e DENSE_RANK:')
run(sql_rank_diff)

Comparação entre ROW_NUMBER, RANK e DENSE_RANK:
{'numero': 'PROP-010', 'score': 950, 'row_number': 1, 'rank': 1, 'dense_rank': 1}
{'numero': 'PROP-004', 'score': 900, 'row_number': 2, 'rank': 2, 'dense_rank': 2}
{'numero': 'PROP-EMPATE-B', 'score': 900, 'row_number': 3, 'rank': 2, 'dense_rank': 2}
{'numero': 'PROP-EMPATE-A', 'score': 900, 'row_number': 4, 'rank': 2, 'dense_rank': 2}
{'numero': 'PROP-014', 'score': 880, 'row_number': 5, 'rank': 5, 'dense_rank': 3}
{'numero': 'PROP-006', 'score': 870, 'row_number': 6, 'rank': 6, 'dense_rank': 4}
{'numero': 'PROP-003', 'score': 810, 'row_number': 7, 'rank': 7, 'dense_rank': 5}
{'numero': 'PROP-008', 'score': 790, 'row_number': 8, 'rank': 8, 'dense_rank': 6}
{'numero': 'PROP-011', 'score': 740, 'row_number': 9, 'rank': 9, 'dense_rank': 7}
{'numero': 'PROP-012', 'score': 730, 'row_number': 10, 'rank': 10, 'dense_rank': 8}


[{'numero': 'PROP-010',
  'score': 950,
  'row_number': 1,
  'rank': 1,
  'dense_rank': 1},
 {'numero': 'PROP-004',
  'score': 900,
  'row_number': 2,
  'rank': 2,
  'dense_rank': 2},
 {'numero': 'PROP-EMPATE-B',
  'score': 900,
  'row_number': 3,
  'rank': 2,
  'dense_rank': 2},
 {'numero': 'PROP-EMPATE-A',
  'score': 900,
  'row_number': 4,
  'rank': 2,
  'dense_rank': 2},
 {'numero': 'PROP-014',
  'score': 880,
  'row_number': 5,
  'rank': 5,
  'dense_rank': 3},
 {'numero': 'PROP-006',
  'score': 870,
  'row_number': 6,
  'rank': 6,
  'dense_rank': 4},
 {'numero': 'PROP-003',
  'score': 810,
  'row_number': 7,
  'rank': 7,
  'dense_rank': 5},
 {'numero': 'PROP-008',
  'score': 790,
  'row_number': 8,
  'rank': 8,
  'dense_rank': 6},
 {'numero': 'PROP-011',
  'score': 740,
  'row_number': 9,
  'rank': 9,
  'dense_rank': 7},
 {'numero': 'PROP-012',
  'score': 730,
  'row_number': 10,
  'rank': 10,
  'dense_rank': 8}]

### 2.3 LAG e LEAD — Comparar com Linha Anterior ou Próxima

`LAG(expr, offset, default)` acessa o valor de uma linha **anterior** dentro da partição.
`LEAD(expr, offset, default)` acessa o valor de uma linha **posterior**.

Caso de uso clássico: evolução mês a mês do volume de propostas — quanto cresceu em relação ao mês anterior?

In [10]:
sql_lag_lead = """
WITH volume_mensal AS (
    SELECT
        DATE_TRUNC('month', created_at)::date AS mes,
        COUNT(*)                              AS qtd_propostas,
        SUM(valor)                            AS volume_total
    FROM propostas
    GROUP BY 1
)
SELECT
    mes,
    qtd_propostas,
    volume_total,
    -- Default 0: para exibição, o primeiro mês mostra "0" como volume anterior
    LAG(volume_total, 1, 0) OVER (ORDER BY mes) AS volume_mes_anterior,
    -- Default volume_total: para o primeiro mês, variação = volume - volume = 0,
    -- evitando NULL e deixando claro que não houve variação (em vez de "sem dado")
    volume_total - LAG(volume_total, 1, volume_total) OVER (ORDER BY mes) AS variacao,
    -- Volume do mês seguinte (LEAD)
    LEAD(volume_total) OVER (ORDER BY mes) AS volume_proximo_mes
FROM volume_mensal
ORDER BY mes;
"""

print('Evolução mensal de volume de propostas:')
run(sql_lag_lead)

Evolução mensal de volume de propostas:
{'mes': datetime.date(2025, 10, 1), 'qtd_propostas': 2, 'volume_total': Decimal('35000.00'), 'volume_mes_anterior': Decimal('0'), 'variacao': Decimal('0.00'), 'volume_proximo_mes': Decimal('95000.00')}
{'mes': datetime.date(2025, 11, 1), 'qtd_propostas': 3, 'volume_total': Decimal('95000.00'), 'volume_mes_anterior': Decimal('35000.00'), 'variacao': Decimal('60000.00'), 'volume_proximo_mes': Decimal('50000.00')}
{'mes': datetime.date(2025, 12, 1), 'qtd_propostas': 1, 'volume_total': Decimal('50000.00'), 'volume_mes_anterior': Decimal('95000.00'), 'variacao': Decimal('-45000.00'), 'volume_proximo_mes': Decimal('129500.00')}
{'mes': datetime.date(2026, 1, 1), 'qtd_propostas': 3, 'volume_total': Decimal('129500.00'), 'volume_mes_anterior': Decimal('50000.00'), 'variacao': Decimal('79500.00'), 'volume_proximo_mes': Decimal('30000.00')}
{'mes': datetime.date(2026, 2, 1), 'qtd_propostas': 3, 'volume_total': Decimal('30000.00'), 'volume_mes_anterior': De

[{'mes': datetime.date(2025, 10, 1),
  'qtd_propostas': 2,
  'volume_total': Decimal('35000.00'),
  'volume_mes_anterior': Decimal('0'),
  'variacao': Decimal('0.00'),
  'volume_proximo_mes': Decimal('95000.00')},
 {'mes': datetime.date(2025, 11, 1),
  'qtd_propostas': 3,
  'volume_total': Decimal('95000.00'),
  'volume_mes_anterior': Decimal('35000.00'),
  'variacao': Decimal('60000.00'),
  'volume_proximo_mes': Decimal('50000.00')},
 {'mes': datetime.date(2025, 12, 1),
  'qtd_propostas': 1,
  'volume_total': Decimal('50000.00'),
  'volume_mes_anterior': Decimal('95000.00'),
  'variacao': Decimal('-45000.00'),
  'volume_proximo_mes': Decimal('129500.00')},
 {'mes': datetime.date(2026, 1, 1),
  'qtd_propostas': 3,
  'volume_total': Decimal('129500.00'),
  'volume_mes_anterior': Decimal('50000.00'),
  'variacao': Decimal('79500.00'),
  'volume_proximo_mes': Decimal('30000.00')},
 {'mes': datetime.date(2026, 2, 1),
  'qtd_propostas': 3,
  'volume_total': Decimal('30000.00'),
  'volume_me

### 2.4 SUM OVER — Running Total (Acumulado)

Com `SUM() OVER (ORDER BY ...)`, o PostgreSQL calcula um **acumulado progressivo**.
O frame padrão para funções de ranking/agregado com ORDER BY é `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`,
o que produz a soma de todas as linhas até a atual.

Útil para: acumulado de aprovações por dia, saldo progressivo, progresso de meta.

In [11]:
sql_running_total = """
WITH aprovadas_por_dia AS (
    SELECT
        created_at::date AS dia,
        SUM(valor)       AS volume_dia
    FROM propostas
    WHERE status = 'aprovada'
    GROUP BY 1
)
SELECT
    dia,
    volume_dia,
    -- Acumulado progressivo: soma de tudo ate o dia atual
    SUM(volume_dia) OVER (
        ORDER BY dia
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS acumulado,
    -- Percentual do acumulado sobre o total geral
    ROUND(
        SUM(volume_dia) OVER (
            ORDER BY dia
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
        / SUM(volume_dia) OVER () * 100
    , 1) AS pct_acumulado
FROM aprovadas_por_dia
ORDER BY dia;
"""

print('Acumulado diário de volume aprovado:')
run(sql_running_total)

Acumulado diário de volume aprovado:
{'dia': datetime.date(2025, 10, 1), 'volume_dia': Decimal('5000.00'), 'acumulado': Decimal('5000.00'), 'pct_acumulado': Decimal('1.3')}
{'dia': datetime.date(2025, 10, 16), 'volume_dia': Decimal('30000.00'), 'acumulado': Decimal('35000.00'), 'pct_acumulado': Decimal('9.2')}
{'dia': datetime.date(2025, 11, 10), 'volume_dia': Decimal('80000.00'), 'acumulado': Decimal('115000.00'), 'pct_acumulado': Decimal('30.1')}
{'dia': datetime.date(2025, 12, 20), 'volume_dia': Decimal('50000.00'), 'acumulado': Decimal('165000.00'), 'pct_acumulado': Decimal('43.2')}
{'dia': datetime.date(2026, 1, 4), 'volume_dia': Decimal('25000.00'), 'acumulado': Decimal('190000.00'), 'pct_acumulado': Decimal('49.7')}
{'dia': datetime.date(2026, 1, 24), 'volume_dia': Decimal('100000.00'), 'acumulado': Decimal('290000.00'), 'pct_acumulado': Decimal('75.9')}
{'dia': datetime.date(2026, 2, 3), 'volume_dia': Decimal('15000.00'), 'acumulado': Decimal('305000.00'), 'pct_acumulado': Deci

[{'dia': datetime.date(2025, 10, 1),
  'volume_dia': Decimal('5000.00'),
  'acumulado': Decimal('5000.00'),
  'pct_acumulado': Decimal('1.3')},
 {'dia': datetime.date(2025, 10, 16),
  'volume_dia': Decimal('30000.00'),
  'acumulado': Decimal('35000.00'),
  'pct_acumulado': Decimal('9.2')},
 {'dia': datetime.date(2025, 11, 10),
  'volume_dia': Decimal('80000.00'),
  'acumulado': Decimal('115000.00'),
  'pct_acumulado': Decimal('30.1')},
 {'dia': datetime.date(2025, 12, 20),
  'volume_dia': Decimal('50000.00'),
  'acumulado': Decimal('165000.00'),
  'pct_acumulado': Decimal('43.2')},
 {'dia': datetime.date(2026, 1, 4),
  'volume_dia': Decimal('25000.00'),
  'acumulado': Decimal('190000.00'),
  'pct_acumulado': Decimal('49.7')},
 {'dia': datetime.date(2026, 1, 24),
  'volume_dia': Decimal('100000.00'),
  'acumulado': Decimal('290000.00'),
  'pct_acumulado': Decimal('75.9')},
 {'dia': datetime.date(2026, 2, 3),
  'volume_dia': Decimal('15000.00'),
  'acumulado': Decimal('305000.00'),
  'pc

### 2.5 COUNT OVER — Total do Grupo sem GROUP BY

Com `COUNT(*) OVER (PARTITION BY ...)`, cada linha recebe o total do seu grupo,
sem colapsar as linhas. Permite calcular percentuais dentro do próprio SELECT.

In [12]:
sql_count_over = """
SELECT
    p.numero,
    p.status,
    p.valor,
    -- Total de propostas neste status (sem GROUP BY)
    COUNT(*) OVER (PARTITION BY p.status)         AS total_no_status,
    -- Percentual desta proposta dentro do status
    ROUND(
        p.valor / SUM(p.valor) OVER (PARTITION BY p.status) * 100
    , 1) AS pct_valor_no_status
FROM propostas p
ORDER BY p.status, p.valor DESC;
"""

print('Propostas com total do grupo e percentual:')
run(sql_count_over)

Propostas com total do grupo e percentual:
{'numero': 'PROP-010', 'status': 'aprovada', 'valor': Decimal('100000.00'), 'total_no_status': 11, 'pct_valor_no_status': Decimal('26.2')}
{'numero': 'PROP-004', 'status': 'aprovada', 'valor': Decimal('80000.00'), 'total_no_status': 11, 'pct_valor_no_status': Decimal('20.9')}
{'numero': 'PROP-014', 'status': 'aprovada', 'valor': Decimal('60000.00'), 'total_no_status': 11, 'pct_valor_no_status': Decimal('15.7')}
{'numero': 'PROP-006', 'status': 'aprovada', 'valor': Decimal('50000.00'), 'total_no_status': 11, 'pct_valor_no_status': Decimal('13.1')}
{'numero': 'PROP-003', 'status': 'aprovada', 'valor': Decimal('30000.00'), 'total_no_status': 11, 'pct_valor_no_status': Decimal('7.9')}
{'numero': 'PROP-008', 'status': 'aprovada', 'valor': Decimal('25000.00'), 'total_no_status': 11, 'pct_valor_no_status': Decimal('6.5')}
{'numero': 'PROP-011', 'status': 'aprovada', 'valor': Decimal('15000.00'), 'total_no_status': 11, 'pct_valor_no_status': Decimal('

[{'numero': 'PROP-010',
  'status': 'aprovada',
  'valor': Decimal('100000.00'),
  'total_no_status': 11,
  'pct_valor_no_status': Decimal('26.2')},
 {'numero': 'PROP-004',
  'status': 'aprovada',
  'valor': Decimal('80000.00'),
  'total_no_status': 11,
  'pct_valor_no_status': Decimal('20.9')},
 {'numero': 'PROP-014',
  'status': 'aprovada',
  'valor': Decimal('60000.00'),
  'total_no_status': 11,
  'pct_valor_no_status': Decimal('15.7')},
 {'numero': 'PROP-006',
  'status': 'aprovada',
  'valor': Decimal('50000.00'),
  'total_no_status': 11,
  'pct_valor_no_status': Decimal('13.1')},
 {'numero': 'PROP-003',
  'status': 'aprovada',
  'valor': Decimal('30000.00'),
  'total_no_status': 11,
  'pct_valor_no_status': Decimal('7.9')},
 {'numero': 'PROP-008',
  'status': 'aprovada',
  'valor': Decimal('25000.00'),
  'total_no_status': 11,
  'pct_valor_no_status': Decimal('6.5')},
 {'numero': 'PROP-011',
  'status': 'aprovada',
  'valor': Decimal('15000.00'),
  'total_no_status': 11,
  'pct_v

### 2.6 NTILE — Dividir Clientes em Quartis por Renda

`NTILE(n)` distribui as linhas em **n grupos de tamanho igual** (ou o mais igual possível).
Perfeito para segmentação: quartis, decis, quintis.

In [13]:
sql_ntile = """
SELECT
    nome,
    renda_mensal,
    NTILE(4) OVER (ORDER BY renda_mensal) AS quartil,
    CASE NTILE(4) OVER (ORDER BY renda_mensal)
        WHEN 1 THEN 'Q1 - Menor renda'
        WHEN 2 THEN 'Q2'
        WHEN 3 THEN 'Q3'
        WHEN 4 THEN 'Q4 - Maior renda'
    END AS segmento
FROM clientes
ORDER BY renda_mensal;
"""

print('Clientes divididos em quartis por renda:')
run(sql_ntile)

Clientes divididos em quartis por renda:
{'nome': 'Diego Rocha', 'renda_mensal': Decimal('2800.00'), 'quartil': 1, 'segmento': 'Q1 - Menor renda'}
{'nome': 'Ana Lima', 'renda_mensal': Decimal('3500.00'), 'quartil': 1, 'segmento': 'Q1 - Menor renda'}
{'nome': 'Henrique Paz', 'renda_mensal': Decimal('4300.00'), 'quartil': 1, 'segmento': 'Q1 - Menor renda'}
{'nome': 'Fabio Nunes', 'renda_mensal': Decimal('5100.00'), 'quartil': 2, 'segmento': 'Q2'}
{'nome': 'Jonas Brito', 'renda_mensal': Decimal('6600.00'), 'quartil': 2, 'segmento': 'Q2'}
{'nome': 'Bruno Souza', 'renda_mensal': Decimal('8200.00'), 'quartil': 2, 'segmento': 'Q2'}
{'nome': 'Gabriela Cruz', 'renda_mensal': Decimal('9700.00'), 'quartil': 3, 'segmento': 'Q3'}
{'nome': 'Carla Mendes', 'renda_mensal': Decimal('15000.00'), 'quartil': 3, 'segmento': 'Q3'}
{'nome': 'Eva Santos', 'renda_mensal': Decimal('22000.00'), 'quartil': 4, 'segmento': 'Q4 - Maior renda'}
{'nome': 'Isabela Teles', 'renda_mensal': Decimal('31000.00'), 'quartil':

[{'nome': 'Diego Rocha',
  'renda_mensal': Decimal('2800.00'),
  'quartil': 1,
  'segmento': 'Q1 - Menor renda'},
 {'nome': 'Ana Lima',
  'renda_mensal': Decimal('3500.00'),
  'quartil': 1,
  'segmento': 'Q1 - Menor renda'},
 {'nome': 'Henrique Paz',
  'renda_mensal': Decimal('4300.00'),
  'quartil': 1,
  'segmento': 'Q1 - Menor renda'},
 {'nome': 'Fabio Nunes',
  'renda_mensal': Decimal('5100.00'),
  'quartil': 2,
  'segmento': 'Q2'},
 {'nome': 'Jonas Brito',
  'renda_mensal': Decimal('6600.00'),
  'quartil': 2,
  'segmento': 'Q2'},
 {'nome': 'Bruno Souza',
  'renda_mensal': Decimal('8200.00'),
  'quartil': 2,
  'segmento': 'Q2'},
 {'nome': 'Gabriela Cruz',
  'renda_mensal': Decimal('9700.00'),
  'quartil': 3,
  'segmento': 'Q3'},
 {'nome': 'Carla Mendes',
  'renda_mensal': Decimal('15000.00'),
  'quartil': 3,
  'segmento': 'Q3'},
 {'nome': 'Eva Santos',
  'renda_mensal': Decimal('22000.00'),
  'quartil': 4,
  'segmento': 'Q4 - Maior renda'},
 {'nome': 'Isabela Teles',
  'renda_mensal

## 3. Subqueries Correlacionadas vs JOINs

Uma **subquery correlacionada** executa uma vez por linha da query externa -- referencia colunas externas, o que a torna dependente.

Um **JOIN** processa o relacionamento de forma global.

**Como decidir:**
1. Rode `EXPLAIN (ANALYZE, BUFFERS)` para ambas as versões
2. Se o plano da subquery mostra `Nested Loop` com `Seq Scan` na tabela interna, o JOIN vai ganhar
3. Se o plano da subquery mostra `Index Scan` na tabela interna e o número de linhas externas é baixo (< 1000), a diferença será negligível
4. Se o otimizador converte a subquery em semi-join (visível como `Hash Semi Join` no EXPLAIN), ambas as formas produzem o mesmo plano -- use a mais legível
5. Em dúvida, prefira JOIN: o custo de leitura é previsível e não depende da cardinalidade linha-a-linha

In [14]:
# Versão com subquery correlacionada:
# Para cada cliente, conta propostas aprovadas referenciando o cliente da linha externa
sql_correlated = """
SELECT
    c.nome,
    c.renda_mensal,
    -- Esta subquery roda uma vez por cliente
    (SELECT COUNT(*)
     FROM propostas p
     WHERE p.cliente_id = c.id
       AND p.status = 'aprovada') AS propostas_aprovadas
FROM clientes c
ORDER BY propostas_aprovadas DESC;
"""

# Versão equivalente com LEFT JOIN + GROUP BY - geralmente mais eficiente
sql_join_equiv = """
SELECT
    c.nome,
    c.renda_mensal,
    COUNT(p.id) AS propostas_aprovadas
FROM clientes c
LEFT JOIN propostas p ON p.cliente_id = c.id AND p.status = 'aprovada'
GROUP BY c.id, c.nome, c.renda_mensal
ORDER BY propostas_aprovadas DESC;
"""

print('Resultado com subquery correlacionada:')
run(sql_correlated)
print()
print('Resultado com JOIN (equivalente):')
run(sql_join_equiv)

Resultado com subquery correlacionada:
{'nome': 'Ana Lima', 'renda_mensal': Decimal('3500.00'), 'propostas_aprovadas': 2}
{'nome': 'Carla Mendes', 'renda_mensal': Decimal('15000.00'), 'propostas_aprovadas': 2}
{'nome': 'Diego Rocha', 'renda_mensal': Decimal('2800.00'), 'propostas_aprovadas': 1}
{'nome': 'Eva Santos', 'renda_mensal': Decimal('22000.00'), 'propostas_aprovadas': 1}
{'nome': 'Fabio Nunes', 'renda_mensal': Decimal('5100.00'), 'propostas_aprovadas': 1}
{'nome': 'Gabriela Cruz', 'renda_mensal': Decimal('9700.00'), 'propostas_aprovadas': 1}
{'nome': 'Isabela Teles', 'renda_mensal': Decimal('31000.00'), 'propostas_aprovadas': 1}
{'nome': 'Jonas Brito', 'renda_mensal': Decimal('6600.00'), 'propostas_aprovadas': 1}
{'nome': 'Bruno Souza', 'renda_mensal': Decimal('8200.00'), 'propostas_aprovadas': 1}
{'nome': 'Henrique Paz', 'renda_mensal': Decimal('4300.00'), 'propostas_aprovadas': 0}

Resultado com JOIN (equivalente):
{'nome': 'Carla Mendes', 'renda_mensal': Decimal('15000.00'),

[{'nome': 'Carla Mendes',
  'renda_mensal': Decimal('15000.00'),
  'propostas_aprovadas': 2},
 {'nome': 'Ana Lima',
  'renda_mensal': Decimal('3500.00'),
  'propostas_aprovadas': 2},
 {'nome': 'Diego Rocha',
  'renda_mensal': Decimal('2800.00'),
  'propostas_aprovadas': 1},
 {'nome': 'Jonas Brito',
  'renda_mensal': Decimal('6600.00'),
  'propostas_aprovadas': 1},
 {'nome': 'Fabio Nunes',
  'renda_mensal': Decimal('5100.00'),
  'propostas_aprovadas': 1},
 {'nome': 'Bruno Souza',
  'renda_mensal': Decimal('8200.00'),
  'propostas_aprovadas': 1},
 {'nome': 'Gabriela Cruz',
  'renda_mensal': Decimal('9700.00'),
  'propostas_aprovadas': 1},
 {'nome': 'Isabela Teles',
  'renda_mensal': Decimal('31000.00'),
  'propostas_aprovadas': 1},
 {'nome': 'Eva Santos',
  'renda_mensal': Decimal('22000.00'),
  'propostas_aprovadas': 1},
 {'nome': 'Henrique Paz',
  'renda_mensal': Decimal('4300.00'),
  'propostas_aprovadas': 0}]

In [15]:
# Comparando os planos de execução
def explain(sql: str) -> None:
    with conn.cursor() as cur:
        cur.execute('EXPLAIN ' + sql)
        for row in cur.fetchall():
            print(list(row.values())[0])

print('=== Plano: subquery correlacionada ===')
explain(sql_correlated)

print()
print('=== Plano: JOIN ===')
explain(sql_join_equiv)

=== Plano: subquery correlacionada ===
Sort  (cost=3708.51..3709.06 rows=220 width=282)
  Sort Key: ((SubPlan 1)) DESC
  ->  Seq Scan on clientes c  (cost=0.00..3699.95 rows=220 width=282)
        SubPlan 1
          ->  Aggregate  (cost=16.75..16.76 rows=1 width=8)
                ->  Seq Scan on propostas p  (cost=0.00..16.75 rows=1 width=0)
                      Filter: ((cliente_id = c.id) AND ((status)::text = 'aprovada'::text))

=== Plano: JOIN ===
Sort  (cost=40.55..41.10 rows=220 width=286)
  Sort Key: (count(p.id)) DESC
  ->  HashAggregate  (cost=29.80..32.00 rows=220 width=286)
        Group Key: c.id
        ->  Hash Left Join  (cost=15.65..28.70 rows=220 width=282)
              Hash Cond: (c.id = p.cliente_id)
              ->  Seq Scan on clientes c  (cost=0.00..12.20 rows=220 width=278)
              ->  Hash  (cost=15.62..15.62 rows=2 width=8)
                    ->  Seq Scan on propostas p  (cost=0.00..15.62 rows=2 width=8)
                          Filter: ((status)::

## 4. LATERAL JOIN — "Para Cada Linha, Buscar N Relacionados"

Um `LATERAL` permite que o subselect à direita referencie colunas da tabela à esquerda,
como uma subquery correlacionada, mas usada como se fosse uma tabela (podendo retornar múltiplas linhas e colunas).

**Caso de uso clássico:** para cada cliente, trazer a última proposta.

**Alternativas e quando cada uma é melhor:**
| Abordagem | Melhor quando |
|---|---|
| LATERAL JOIN | Precisa de múltiplas colunas do registro relacionado mais recente |
| Window Function + CTE | Precisa de N registros por grupo com filtro posterior |
| Subquery correlacionada | Apenas um campo do registro relacionado |

In [16]:
# Abordagem 1: LATERAL JOIN
# Para cada cliente, busca a proposta mais recente com todas as suas colunas
sql_lateral = """
SELECT
    c.nome,
    c.renda_mensal,
    ultima.numero        AS ultima_proposta,
    ultima.valor         AS valor,
    ultima.status        AS status,
    ultima.created_at    AS data
FROM clientes c
LEFT JOIN LATERAL (
    SELECT numero, valor, status, created_at
    FROM propostas p
    WHERE p.cliente_id = c.id   -- referencia a linha atual de clientes
    ORDER BY p.created_at DESC
    LIMIT 1
) ultima ON true
ORDER BY c.nome;
"""

print('Ultima proposta de cada cliente (LATERAL JOIN):')
run(sql_lateral)

Ultima proposta de cada cliente (LATERAL JOIN):
{'nome': 'Ana Lima', 'renda_mensal': Decimal('3500.00'), 'ultima_proposta': 'PROP-012', 'valor': Decimal('7000.00'), 'status': 'aprovada', 'data': datetime.datetime(2026, 2, 18, 13, 15, 46, 904302, tzinfo=datetime.timezone.utc)}
{'nome': 'Bruno Souza', 'renda_mensal': Decimal('8200.00'), 'ultima_proposta': 'PROP-013', 'valor': Decimal('20000.00'), 'status': 'pendente', 'data': datetime.datetime(2026, 3, 10, 13, 15, 46, 904302, tzinfo=datetime.timezone.utc)}
{'nome': 'Carla Mendes', 'renda_mensal': Decimal('15000.00'), 'ultima_proposta': 'PROP-014', 'valor': Decimal('60000.00'), 'status': 'aprovada', 'data': datetime.datetime(2026, 3, 15, 13, 15, 46, 904302, tzinfo=datetime.timezone.utc)}
{'nome': 'Diego Rocha', 'renda_mensal': Decimal('2800.00'), 'ultima_proposta': 'PROP-EMPATE-A', 'valor': Decimal('5000.00'), 'status': 'aprovada', 'data': datetime.datetime(2026, 3, 20, 13, 15, 46, 976842, tzinfo=datetime.timezone.utc)}
{'nome': 'Eva Sant

[{'nome': 'Ana Lima',
  'renda_mensal': Decimal('3500.00'),
  'ultima_proposta': 'PROP-012',
  'valor': Decimal('7000.00'),
  'status': 'aprovada',
  'data': datetime.datetime(2026, 2, 18, 13, 15, 46, 904302, tzinfo=datetime.timezone.utc)},
 {'nome': 'Bruno Souza',
  'renda_mensal': Decimal('8200.00'),
  'ultima_proposta': 'PROP-013',
  'valor': Decimal('20000.00'),
  'status': 'pendente',
  'data': datetime.datetime(2026, 3, 10, 13, 15, 46, 904302, tzinfo=datetime.timezone.utc)},
 {'nome': 'Carla Mendes',
  'renda_mensal': Decimal('15000.00'),
  'ultima_proposta': 'PROP-014',
  'valor': Decimal('60000.00'),
  'status': 'aprovada',
  'data': datetime.datetime(2026, 3, 15, 13, 15, 46, 904302, tzinfo=datetime.timezone.utc)},
 {'nome': 'Diego Rocha',
  'renda_mensal': Decimal('2800.00'),
  'ultima_proposta': 'PROP-EMPATE-A',
  'valor': Decimal('5000.00'),
  'status': 'aprovada',
  'data': datetime.datetime(2026, 3, 20, 13, 15, 46, 976842, tzinfo=datetime.timezone.utc)},
 {'nome': 'Eva San

In [17]:
# Abordagem 2: Window Function + CTE (alternativa ao LATERAL)
sql_window_equiv = """
WITH propostas_ranked AS (
    SELECT
        cliente_id,
        numero,
        valor,
        status,
        created_at,
        ROW_NUMBER() OVER (PARTITION BY cliente_id ORDER BY created_at DESC) AS rn
    FROM propostas
)
SELECT
    c.nome,
    c.renda_mensal,
    pr.numero        AS ultima_proposta,
    pr.valor,
    pr.status,
    pr.created_at    AS data
FROM clientes c
LEFT JOIN propostas_ranked pr ON pr.cliente_id = c.id AND pr.rn = 1
ORDER BY c.nome;
"""

print('Ultima proposta de cada cliente (Window Function - equivalente):')
run(sql_window_equiv)

Ultima proposta de cada cliente (Window Function - equivalente):
{'nome': 'Ana Lima', 'renda_mensal': Decimal('3500.00'), 'ultima_proposta': 'PROP-012', 'valor': Decimal('7000.00'), 'status': 'aprovada', 'data': datetime.datetime(2026, 2, 18, 13, 15, 46, 904302, tzinfo=datetime.timezone.utc)}
{'nome': 'Bruno Souza', 'renda_mensal': Decimal('8200.00'), 'ultima_proposta': 'PROP-013', 'valor': Decimal('20000.00'), 'status': 'pendente', 'data': datetime.datetime(2026, 3, 10, 13, 15, 46, 904302, tzinfo=datetime.timezone.utc)}
{'nome': 'Carla Mendes', 'renda_mensal': Decimal('15000.00'), 'ultima_proposta': 'PROP-014', 'valor': Decimal('60000.00'), 'status': 'aprovada', 'data': datetime.datetime(2026, 3, 15, 13, 15, 46, 904302, tzinfo=datetime.timezone.utc)}
{'nome': 'Diego Rocha', 'renda_mensal': Decimal('2800.00'), 'ultima_proposta': 'PROP-EMPATE-A', 'valor': Decimal('5000.00'), 'status': 'aprovada', 'data': datetime.datetime(2026, 3, 20, 13, 15, 46, 976842, tzinfo=datetime.timezone.utc)}
{

[{'nome': 'Ana Lima',
  'renda_mensal': Decimal('3500.00'),
  'ultima_proposta': 'PROP-012',
  'valor': Decimal('7000.00'),
  'status': 'aprovada',
  'data': datetime.datetime(2026, 2, 18, 13, 15, 46, 904302, tzinfo=datetime.timezone.utc)},
 {'nome': 'Bruno Souza',
  'renda_mensal': Decimal('8200.00'),
  'ultima_proposta': 'PROP-013',
  'valor': Decimal('20000.00'),
  'status': 'pendente',
  'data': datetime.datetime(2026, 3, 10, 13, 15, 46, 904302, tzinfo=datetime.timezone.utc)},
 {'nome': 'Carla Mendes',
  'renda_mensal': Decimal('15000.00'),
  'ultima_proposta': 'PROP-014',
  'valor': Decimal('60000.00'),
  'status': 'aprovada',
  'data': datetime.datetime(2026, 3, 15, 13, 15, 46, 904302, tzinfo=datetime.timezone.utc)},
 {'nome': 'Diego Rocha',
  'renda_mensal': Decimal('2800.00'),
  'ultima_proposta': 'PROP-EMPATE-A',
  'valor': Decimal('5000.00'),
  'status': 'aprovada',
  'data': datetime.datetime(2026, 3, 20, 13, 15, 46, 976842, tzinfo=datetime.timezone.utc)},
 {'nome': 'Eva San

### 4.1 LATERAL para Top-N por Grupo

Enquanto window function + CTE requer filtrar `rn <= N` na query externa,
o LATERAL encapsula o `LIMIT N` diretamente e é frequentemente mais claro para esse padrão.

In [18]:
# Para cada cliente, trazer as 2 propostas de maior valor
# LEFT JOIN LATERAL preserva clientes sem propostas (consistente com cell-33)
sql_lateral_top2 = """
SELECT
    c.nome,
    top2.numero,
    top2.valor,
    top2.status
FROM clientes c
LEFT JOIN LATERAL (
    SELECT numero, valor, status
    FROM propostas p
    WHERE p.cliente_id = c.id
    ORDER BY p.valor DESC
    LIMIT 2
) top2 ON true
ORDER BY c.nome, top2.valor DESC;
"""

print('Top 2 propostas por valor para cada cliente:')
run(sql_lateral_top2)

Top 2 propostas por valor para cada cliente:
{'nome': 'Ana Lima', 'numero': 'PROP-002', 'valor': Decimal('12000.00'), 'status': 'reprovada'}
{'nome': 'Ana Lima', 'numero': 'PROP-012', 'valor': Decimal('7000.00'), 'status': 'aprovada'}
{'nome': 'Bruno Souza', 'numero': 'PROP-003', 'valor': Decimal('30000.00'), 'status': 'aprovada'}
{'nome': 'Bruno Souza', 'numero': 'PROP-013', 'valor': Decimal('20000.00'), 'status': 'pendente'}
{'nome': 'Carla Mendes', 'numero': 'PROP-004', 'valor': Decimal('80000.00'), 'status': 'aprovada'}
{'nome': 'Carla Mendes', 'numero': 'PROP-014', 'valor': Decimal('60000.00'), 'status': 'aprovada'}
{'nome': 'Diego Rocha', 'numero': 'PROP-EMPATE-A', 'valor': Decimal('5000.00'), 'status': 'aprovada'}
{'nome': 'Diego Rocha', 'numero': 'PROP-005', 'valor': Decimal('3000.00'), 'status': 'reprovada'}
{'nome': 'Eva Santos', 'numero': 'PROP-006', 'valor': Decimal('50000.00'), 'status': 'aprovada'}
{'nome': 'Fabio Nunes', 'numero': 'PROP-007', 'valor': Decimal('8000.00'),

[{'nome': 'Ana Lima',
  'numero': 'PROP-002',
  'valor': Decimal('12000.00'),
  'status': 'reprovada'},
 {'nome': 'Ana Lima',
  'numero': 'PROP-012',
  'valor': Decimal('7000.00'),
  'status': 'aprovada'},
 {'nome': 'Bruno Souza',
  'numero': 'PROP-003',
  'valor': Decimal('30000.00'),
  'status': 'aprovada'},
 {'nome': 'Bruno Souza',
  'numero': 'PROP-013',
  'valor': Decimal('20000.00'),
  'status': 'pendente'},
 {'nome': 'Carla Mendes',
  'numero': 'PROP-004',
  'valor': Decimal('80000.00'),
  'status': 'aprovada'},
 {'nome': 'Carla Mendes',
  'numero': 'PROP-014',
  'valor': Decimal('60000.00'),
  'status': 'aprovada'},
 {'nome': 'Diego Rocha',
  'numero': 'PROP-EMPATE-A',
  'valor': Decimal('5000.00'),
  'status': 'aprovada'},
 {'nome': 'Diego Rocha',
  'numero': 'PROP-005',
  'valor': Decimal('3000.00'),
  'status': 'reprovada'},
 {'nome': 'Eva Santos',
  'numero': 'PROP-006',
  'valor': Decimal('50000.00'),
  'status': 'aprovada'},
 {'nome': 'Fabio Nunes',
  'numero': 'PROP-007'

## 5. Agregações Condicionais

Em vez de múltiplos JOINs com subqueries por status, agregações condicionais calculam
contagens e somas para diferentes condições em um único GROUP BY.

**Duas sintaxes equivalentes:**
- `COUNT(*) FILTER (WHERE condicao)` — padrão SQL:2003, mais limpo
- `SUM(CASE WHEN condicao THEN 1 ELSE 0 END)` — compatível com bancos mais antigos

**Quando usar:** relatórios pivot-like onde cada coluna representa um subconjunto diferente da mesma tabela.

In [19]:
sql_conditional_agg = """
SELECT
    DATE_TRUNC('month', p.created_at)::date     AS mes,

    -- Contagem por status com FILTER (sintaxe moderna)
    COUNT(*) FILTER (WHERE p.status = 'aprovada')   AS aprovadas,
    COUNT(*) FILTER (WHERE p.status = 'reprovada')  AS reprovadas,
    COUNT(*) FILTER (WHERE p.status = 'pendente')   AS pendentes,
    COUNT(*) FILTER (WHERE p.status = 'cancelada')  AS canceladas,
    COUNT(*)                                         AS total,

    -- Soma condicional com CASE WHEN (compatibilidade)
    SUM(CASE WHEN p.status = 'aprovada' THEN p.valor ELSE 0 END) AS volume_aprovado,

    -- Taxa de aprovação (evita divisão por zero com NULLIF)
    ROUND(
        COUNT(*) FILTER (WHERE p.status = 'aprovada')::numeric
        / NULLIF(COUNT(*), 0) * 100
    , 1) AS taxa_aprovacao_pct

FROM propostas p
GROUP BY 1
ORDER BY 1;
"""

print('Resumo mensal com agregações condicionais:')
run(sql_conditional_agg)

Resumo mensal com agregações condicionais:
{'mes': datetime.date(2025, 10, 1), 'aprovadas': 2, 'reprovadas': 0, 'pendentes': 0, 'canceladas': 0, 'total': 2, 'volume_aprovado': Decimal('35000.00'), 'taxa_aprovacao_pct': Decimal('100.0')}
{'mes': datetime.date(2025, 11, 1), 'aprovadas': 1, 'reprovadas': 2, 'pendentes': 0, 'canceladas': 0, 'total': 3, 'volume_aprovado': Decimal('80000.00'), 'taxa_aprovacao_pct': Decimal('33.3')}
{'mes': datetime.date(2025, 12, 1), 'aprovadas': 1, 'reprovadas': 0, 'pendentes': 0, 'canceladas': 0, 'total': 1, 'volume_aprovado': Decimal('50000.00'), 'taxa_aprovacao_pct': Decimal('100.0')}
{'mes': datetime.date(2026, 1, 1), 'aprovadas': 2, 'reprovadas': 0, 'pendentes': 0, 'canceladas': 1, 'total': 3, 'volume_aprovado': Decimal('125000.00'), 'taxa_aprovacao_pct': Decimal('66.7')}
{'mes': datetime.date(2026, 2, 1), 'aprovadas': 2, 'reprovadas': 0, 'pendentes': 1, 'canceladas': 0, 'total': 3, 'volume_aprovado': Decimal('22000.00'), 'taxa_aprovacao_pct': Decimal(

[{'mes': datetime.date(2025, 10, 1),
  'aprovadas': 2,
  'reprovadas': 0,
  'pendentes': 0,
  'canceladas': 0,
  'total': 2,
  'volume_aprovado': Decimal('35000.00'),
  'taxa_aprovacao_pct': Decimal('100.0')},
 {'mes': datetime.date(2025, 11, 1),
  'aprovadas': 1,
  'reprovadas': 2,
  'pendentes': 0,
  'canceladas': 0,
  'total': 3,
  'volume_aprovado': Decimal('80000.00'),
  'taxa_aprovacao_pct': Decimal('33.3')},
 {'mes': datetime.date(2025, 12, 1),
  'aprovadas': 1,
  'reprovadas': 0,
  'pendentes': 0,
  'canceladas': 0,
  'total': 1,
  'volume_aprovado': Decimal('50000.00'),
  'taxa_aprovacao_pct': Decimal('100.0')},
 {'mes': datetime.date(2026, 1, 1),
  'aprovadas': 2,
  'reprovadas': 0,
  'pendentes': 0,
  'canceladas': 1,
  'total': 3,
  'volume_aprovado': Decimal('125000.00'),
  'taxa_aprovacao_pct': Decimal('66.7')},
 {'mes': datetime.date(2026, 2, 1),
  'aprovadas': 2,
  'reprovadas': 0,
  'pendentes': 1,
  'canceladas': 0,
  'total': 3,
  'volume_aprovado': Decimal('22000.00

In [20]:
# Pivot por faixa de renda x status - sem CROSSTAB
sql_pivot = """
SELECT
    CASE
        WHEN c.renda_mensal < 5000  THEN 'Ate R$5k'
        WHEN c.renda_mensal < 15000 THEN 'R$5k a R$15k'
        ELSE 'Acima de R$15k'
    END AS faixa_renda,

    COUNT(*) FILTER (WHERE p.status = 'aprovada')  AS aprovadas,
    COUNT(*) FILTER (WHERE p.status = 'reprovada') AS reprovadas,
    COUNT(*) FILTER (WHERE p.status = 'pendente')  AS pendentes,

    ROUND(
        COUNT(*) FILTER (WHERE p.status = 'aprovada')::numeric
        / NULLIF(COUNT(*), 0) * 100
    , 1) AS taxa_aprovacao_pct

FROM propostas p
JOIN clientes c ON c.id = p.cliente_id
GROUP BY 1
ORDER BY MIN(c.renda_mensal);
"""

print('Taxa de aprovação por faixa de renda (pivot manual):')
run(sql_pivot)

Taxa de aprovação por faixa de renda (pivot manual):
{'faixa_renda': 'Ate R$5k', 'aprovadas': 3, 'reprovadas': 2, 'pendentes': 0, 'taxa_aprovacao_pct': Decimal('50.0')}
{'faixa_renda': 'R$5k a R$15k', 'aprovadas': 4, 'reprovadas': 0, 'pendentes': 2, 'taxa_aprovacao_pct': Decimal('66.7')}
{'faixa_renda': 'Acima de R$15k', 'aprovadas': 4, 'reprovadas': 0, 'pendentes': 0, 'taxa_aprovacao_pct': Decimal('100.0')}


[{'faixa_renda': 'Ate R$5k',
  'aprovadas': 3,
  'reprovadas': 2,
  'pendentes': 0,
  'taxa_aprovacao_pct': Decimal('50.0')},
 {'faixa_renda': 'R$5k a R$15k',
  'aprovadas': 4,
  'reprovadas': 0,
  'pendentes': 2,
  'taxa_aprovacao_pct': Decimal('66.7')},
 {'faixa_renda': 'Acima de R$15k',
  'aprovadas': 4,
  'reprovadas': 0,
  'pendentes': 0,
  'taxa_aprovacao_pct': Decimal('100.0')}]

## 6. Caso Real: Dashboard do Gestor

O gestor de crédito precisa de um painel com quatro visões:
1. Total de propostas por status no período
2. Ticket médio por mês
3. Top 10 clientes por valor total aprovado
4. Taxa de aprovação por faixa de renda

Cada query é construída passo a passo, combinando as técnicas do notebook.

In [21]:
# ---- Visão 1: Total de propostas por status ----
sql_v1 = """
SELECT
    status,
    COUNT(*)   AS quantidade,
    SUM(valor) AS volume_total,
    ROUND(AVG(valor), 2) AS ticket_medio
FROM propostas
GROUP BY status
ORDER BY quantidade DESC;
"""

print('Visão 1 -- Total por status:')
run(sql_v1)

Visão 1 -- Total por status:
{'status': 'aprovada', 'quantidade': 11, 'volume_total': Decimal('382000.00'), 'ticket_medio': Decimal('34727.27')}
{'status': 'reprovada', 'quantidade': 2, 'volume_total': Decimal('15000.00'), 'ticket_medio': Decimal('7500.00')}
{'status': 'pendente', 'quantidade': 2, 'volume_total': Decimal('28000.00'), 'ticket_medio': Decimal('14000.00')}
{'status': 'cancelada', 'quantidade': 1, 'volume_total': Decimal('4500.00'), 'ticket_medio': Decimal('4500.00')}


[{'status': 'aprovada',
  'quantidade': 11,
  'volume_total': Decimal('382000.00'),
  'ticket_medio': Decimal('34727.27')},
 {'status': 'reprovada',
  'quantidade': 2,
  'volume_total': Decimal('15000.00'),
  'ticket_medio': Decimal('7500.00')},
 {'status': 'pendente',
  'quantidade': 2,
  'volume_total': Decimal('28000.00'),
  'ticket_medio': Decimal('14000.00')},
 {'status': 'cancelada',
  'quantidade': 1,
  'volume_total': Decimal('4500.00'),
  'ticket_medio': Decimal('4500.00')}]

In [22]:
# ---- Visão 2: Ticket médio por mês com variação em relação ao mês anterior ----
sql_v2 = """
WITH mensal AS (
    SELECT
        DATE_TRUNC('month', created_at)::date AS mes,
        COUNT(*)                              AS qtd,
        ROUND(AVG(valor), 2)                  AS ticket_medio
    FROM propostas
    WHERE status = 'aprovada'
    GROUP BY 1
)
SELECT
    mes,
    qtd,
    ticket_medio,
    LAG(ticket_medio) OVER (ORDER BY mes)                        AS ticket_mes_anterior,
    ROUND(
        ticket_medio - LAG(ticket_medio) OVER (ORDER BY mes)
    , 2) AS variacao_absoluta
FROM mensal
ORDER BY mes;
"""

print('Visão 2 -- Ticket médio mensal com evolução:')
run(sql_v2)

Visão 2 -- Ticket médio mensal com evolução:
{'mes': datetime.date(2025, 10, 1), 'qtd': 2, 'ticket_medio': Decimal('17500.00'), 'ticket_mes_anterior': None, 'variacao_absoluta': None}
{'mes': datetime.date(2025, 11, 1), 'qtd': 1, 'ticket_medio': Decimal('80000.00'), 'ticket_mes_anterior': Decimal('17500.00'), 'variacao_absoluta': Decimal('62500.00')}
{'mes': datetime.date(2025, 12, 1), 'qtd': 1, 'ticket_medio': Decimal('50000.00'), 'ticket_mes_anterior': Decimal('80000.00'), 'variacao_absoluta': Decimal('-30000.00')}
{'mes': datetime.date(2026, 1, 1), 'qtd': 2, 'ticket_medio': Decimal('62500.00'), 'ticket_mes_anterior': Decimal('50000.00'), 'variacao_absoluta': Decimal('12500.00')}
{'mes': datetime.date(2026, 2, 1), 'qtd': 2, 'ticket_medio': Decimal('11000.00'), 'ticket_mes_anterior': Decimal('62500.00'), 'variacao_absoluta': Decimal('-51500.00')}
{'mes': datetime.date(2026, 3, 1), 'qtd': 3, 'ticket_medio': Decimal('23333.33'), 'ticket_mes_anterior': Decimal('11000.00'), 'variacao_abso

[{'mes': datetime.date(2025, 10, 1),
  'qtd': 2,
  'ticket_medio': Decimal('17500.00'),
  'ticket_mes_anterior': None,
  'variacao_absoluta': None},
 {'mes': datetime.date(2025, 11, 1),
  'qtd': 1,
  'ticket_medio': Decimal('80000.00'),
  'ticket_mes_anterior': Decimal('17500.00'),
  'variacao_absoluta': Decimal('62500.00')},
 {'mes': datetime.date(2025, 12, 1),
  'qtd': 1,
  'ticket_medio': Decimal('50000.00'),
  'ticket_mes_anterior': Decimal('80000.00'),
  'variacao_absoluta': Decimal('-30000.00')},
 {'mes': datetime.date(2026, 1, 1),
  'qtd': 2,
  'ticket_medio': Decimal('62500.00'),
  'ticket_mes_anterior': Decimal('50000.00'),
  'variacao_absoluta': Decimal('12500.00')},
 {'mes': datetime.date(2026, 2, 1),
  'qtd': 2,
  'ticket_medio': Decimal('11000.00'),
  'ticket_mes_anterior': Decimal('62500.00'),
  'variacao_absoluta': Decimal('-51500.00')},
 {'mes': datetime.date(2026, 3, 1),
  'qtd': 3,
  'ticket_medio': Decimal('23333.33'),
  'ticket_mes_anterior': Decimal('11000.00'),
  

In [23]:
# ---- Visão 3: Top 10 clientes por volume aprovado ----
sql_v3 = """
WITH volume_cliente AS (
    SELECT
        p.cliente_id,
        COUNT(*)   AS total_propostas,
        SUM(valor) AS volume_aprovado
    FROM propostas p
    WHERE p.status = 'aprovada'
    GROUP BY p.cliente_id
),
ranked AS (
    SELECT
        c.nome,
        c.renda_mensal,
        vc.total_propostas,
        vc.volume_aprovado,
        -- Percentual sobre o total geral aprovado
        ROUND(
            vc.volume_aprovado / SUM(vc.volume_aprovado) OVER () * 100
        , 1) AS pct_do_total,
        -- Acumulado dos melhores clientes
        SUM(vc.volume_aprovado) OVER (
            ORDER BY vc.volume_aprovado DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS acumulado,
        ROW_NUMBER() OVER (ORDER BY vc.volume_aprovado DESC) AS posicao
    FROM volume_cliente vc
    JOIN clientes c ON c.id = vc.cliente_id
)
SELECT posicao, nome, renda_mensal, total_propostas, volume_aprovado, pct_do_total, acumulado
FROM ranked
WHERE posicao <= 10
ORDER BY posicao;
"""

print('Visão 3 -- Top 10 clientes por volume aprovado:')
run(sql_v3)

Visão 3 -- Top 10 clientes por volume aprovado:
{'posicao': 1, 'nome': 'Carla Mendes', 'renda_mensal': Decimal('15000.00'), 'total_propostas': 2, 'volume_aprovado': Decimal('140000.00'), 'pct_do_total': Decimal('36.6'), 'acumulado': Decimal('140000.00')}
{'posicao': 2, 'nome': 'Isabela Teles', 'renda_mensal': Decimal('31000.00'), 'total_propostas': 1, 'volume_aprovado': Decimal('100000.00'), 'pct_do_total': Decimal('26.2'), 'acumulado': Decimal('240000.00')}
{'posicao': 3, 'nome': 'Eva Santos', 'renda_mensal': Decimal('22000.00'), 'total_propostas': 1, 'volume_aprovado': Decimal('50000.00'), 'pct_do_total': Decimal('13.1'), 'acumulado': Decimal('290000.00')}
{'posicao': 4, 'nome': 'Bruno Souza', 'renda_mensal': Decimal('8200.00'), 'total_propostas': 1, 'volume_aprovado': Decimal('30000.00'), 'pct_do_total': Decimal('7.9'), 'acumulado': Decimal('320000.00')}
{'posicao': 5, 'nome': 'Gabriela Cruz', 'renda_mensal': Decimal('9700.00'), 'total_propostas': 1, 'volume_aprovado': Decimal('2500

[{'posicao': 1,
  'nome': 'Carla Mendes',
  'renda_mensal': Decimal('15000.00'),
  'total_propostas': 2,
  'volume_aprovado': Decimal('140000.00'),
  'pct_do_total': Decimal('36.6'),
  'acumulado': Decimal('140000.00')},
 {'posicao': 2,
  'nome': 'Isabela Teles',
  'renda_mensal': Decimal('31000.00'),
  'total_propostas': 1,
  'volume_aprovado': Decimal('100000.00'),
  'pct_do_total': Decimal('26.2'),
  'acumulado': Decimal('240000.00')},
 {'posicao': 3,
  'nome': 'Eva Santos',
  'renda_mensal': Decimal('22000.00'),
  'total_propostas': 1,
  'volume_aprovado': Decimal('50000.00'),
  'pct_do_total': Decimal('13.1'),
  'acumulado': Decimal('290000.00')},
 {'posicao': 4,
  'nome': 'Bruno Souza',
  'renda_mensal': Decimal('8200.00'),
  'total_propostas': 1,
  'volume_aprovado': Decimal('30000.00'),
  'pct_do_total': Decimal('7.9'),
  'acumulado': Decimal('320000.00')},
 {'posicao': 5,
  'nome': 'Gabriela Cruz',
  'renda_mensal': Decimal('9700.00'),
  'total_propostas': 1,
  'volume_aprovad

In [24]:
# ---- Visão 4: Taxa de aprovação por faixa de renda com quartil ----
sql_v4 = """
WITH clientes_quartil AS (
    SELECT
        id,
        nome,
        renda_mensal,
        NTILE(4) OVER (ORDER BY renda_mensal) AS quartil_renda
    FROM clientes
),
metricas AS (
    SELECT
        cq.quartil_renda,
        MIN(cq.renda_mensal)  AS renda_min,
        MAX(cq.renda_mensal)  AS renda_max,
        COUNT(p.id)           AS total_propostas,
        COUNT(p.id) FILTER (WHERE p.status = 'aprovada')  AS aprovadas,
        COUNT(p.id) FILTER (WHERE p.status = 'reprovada') AS reprovadas,
        -- score é NOT NULL no DDL, FILTER desnecessário
        ROUND(AVG(p.score), 0) AS score_medio
    FROM clientes_quartil cq
    LEFT JOIN propostas p ON p.cliente_id = cq.id
    GROUP BY cq.quartil_renda
)
SELECT
    quartil_renda,
    renda_min,
    renda_max,
    total_propostas,
    aprovadas,
    reprovadas,
    score_medio,
    ROUND(
        aprovadas::numeric / NULLIF(total_propostas, 0) * 100
    , 1) AS taxa_aprovacao_pct
FROM metricas
ORDER BY quartil_renda;
"""

print('Visão 4 -- Taxa de aprovação por quartil de renda:')
run(sql_v4)

Visão 4 -- Taxa de aprovação por quartil de renda:
{'quartil_renda': 1, 'renda_min': Decimal('2800.00'), 'renda_max': Decimal('4300.00'), 'total_propostas': 6, 'aprovadas': 3, 'reprovadas': 2, 'score_medio': Decimal('652'), 'taxa_aprovacao_pct': Decimal('50.0')}
{'quartil_renda': 2, 'renda_min': Decimal('5100.00'), 'renda_max': Decimal('8200.00'), 'total_propostas': 5, 'aprovadas': 3, 'reprovadas': 0, 'score_medio': Decimal('790'), 'taxa_aprovacao_pct': Decimal('60.0')}
{'quartil_renda': 3, 'renda_min': Decimal('9700.00'), 'renda_max': Decimal('15000.00'), 'total_propostas': 3, 'aprovadas': 3, 'reprovadas': 0, 'score_medio': Decimal('857'), 'taxa_aprovacao_pct': Decimal('100.0')}
{'quartil_renda': 4, 'renda_min': Decimal('22000.00'), 'renda_max': Decimal('31000.00'), 'total_propostas': 2, 'aprovadas': 2, 'reprovadas': 0, 'score_medio': Decimal('910'), 'taxa_aprovacao_pct': Decimal('100.0')}


[{'quartil_renda': 1,
  'renda_min': Decimal('2800.00'),
  'renda_max': Decimal('4300.00'),
  'total_propostas': 6,
  'aprovadas': 3,
  'reprovadas': 2,
  'score_medio': Decimal('652'),
  'taxa_aprovacao_pct': Decimal('50.0')},
 {'quartil_renda': 2,
  'renda_min': Decimal('5100.00'),
  'renda_max': Decimal('8200.00'),
  'total_propostas': 5,
  'aprovadas': 3,
  'reprovadas': 0,
  'score_medio': Decimal('790'),
  'taxa_aprovacao_pct': Decimal('60.0')},
 {'quartil_renda': 3,
  'renda_min': Decimal('9700.00'),
  'renda_max': Decimal('15000.00'),
  'total_propostas': 3,
  'aprovadas': 3,
  'reprovadas': 0,
  'score_medio': Decimal('857'),
  'taxa_aprovacao_pct': Decimal('100.0')},
 {'quartil_renda': 4,
  'renda_min': Decimal('22000.00'),
  'renda_max': Decimal('31000.00'),
  'total_propostas': 2,
  'aprovadas': 2,
  'reprovadas': 0,
  'score_medio': Decimal('910'),
  'taxa_aprovacao_pct': Decimal('100.0')}]

## Cheat Sheet

**CTE (`WITH`)** -- Use quando a mesma lógica intermediária aparece mais de uma vez, ou quando o aninhamento passa de dois níveis. `WITH RECURSIVE` resolve hierarquias (árvores, grafos acíclicos). O PostgreSQL faz inline por padrão; force `MATERIALIZED` se precisar isolar o plano.

**Window Functions** -- Executam após WHERE/JOIN, antes de ORDER BY final. Para filtrar pelo resultado, envolva em CTE ou subquery.

| Função | Quando usar |
|---|---|
| `ROW_NUMBER` | Top-N por grupo, deduplicação |
| `RANK / DENSE_RANK` | Ranking com empates (RANK pula posição, DENSE_RANK não) |
| `LAG(col, N, default)` | Comparar com linha anterior -- cuidado com o default: 0 para exibição, valor próprio para zerar variação |
| `SUM/COUNT OVER (ORDER BY ...)` | Running total. Frame padrão com ORDER BY é `RANGE UNBOUNDED PRECEDING`, mas prefira `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` para ser explícito |
| `NTILE(n)` | Segmentação em quartis/decis. Trivial se n >= número de linhas |

**Subquery correlacionada vs JOIN** -- Rode `EXPLAIN (ANALYZE, BUFFERS)`. Se o plano mostra Nested Loop + Seq Scan na tabela interna, reescreva como JOIN. Se mostra Index Scan com poucas linhas externas, tanto faz.

**LATERAL JOIN** -- Subselect que referencia colunas da tabela à esquerda. Use `LEFT JOIN LATERAL ... ON true` para não perder linhas sem match. Padrão clássico: `LIMIT N` dentro do LATERAL para top-N por grupo.

**Agregação condicional** -- `COUNT(*) FILTER (WHERE ...)` substitui múltiplos JOINs para pivotar dados. `SUM(CASE WHEN ... END)` é equivalente e roda em bancos mais antigos.

## Exercícios

**Exercício 1 -- Ranking de inadimplência por cliente**

Para cada cliente com propostas aprovadas, calcule: total de parcelas, parcelas atrasadas, taxa de inadimplência, e ranking da maior para a menor taxa.

---

**Exercício 2 -- Evolução do score médio mês a mês**

Score médio das propostas aprovadas por mês, com `score_mes_anterior` (LAG), `variacao_score` (diferença) e `media_movel_acumulada` (AVG OVER).

---

**Exercício 3 -- Top 3 propostas por cliente usando LATERAL**

Para cada cliente, as 3 propostas com maior score: nome do cliente, número da proposta, score, status e posição (1, 2 ou 3).

---

**Exercício 4 -- Distribuição de aprovações em quartis de score**

Divida os clientes em 4 grupos (quartis) pelo score médio das suas propostas. Para cada quartil: score mínimo e máximo do grupo, quantidade de clientes, taxa de aprovação.

**Nota:** o seed tem apenas 10 clientes. Com NTILE(4) o resultado já é útil (2-3 clientes por quartil). Para validar com volume real, use o seed de 200k clientes do notebook 01.

---

**Exercício 5 -- Relatório completo: primeira e última proposta por cliente**

Para cada cliente, em uma única linha: nome, renda, dados da primeira proposta (número, valor, status), dados da última proposta, e variação de valor entre primeira e última.

## Respostas

In [25]:
# Exercício 1 -- Ranking de inadimplência por cliente
# CTE calcula métricas de parcelas por cliente, window function faz o ranking.
# LEFT JOIN em parcelas garante que clientes aprovados sem parcelas (improvável mas seguro) aparecem com 0.

sql_ex1 = """
WITH inadimplencia AS (
    SELECT
        c.nome,
        COUNT(par.id)                                          AS total_parcelas,
        COUNT(par.id) FILTER (WHERE par.status_pagamento = 'atrasada') AS parcelas_atrasadas,
        ROUND(
            COUNT(par.id) FILTER (WHERE par.status_pagamento = 'atrasada')::numeric
            / NULLIF(COUNT(par.id), 0) * 100
        , 1) AS taxa_inadimplencia
    FROM clientes c
    JOIN propostas p ON p.cliente_id = c.id AND p.status = 'aprovada'
    LEFT JOIN parcelas par ON par.proposta_id = p.id
    GROUP BY c.id, c.nome
)
SELECT
    nome,
    total_parcelas,
    parcelas_atrasadas,
    taxa_inadimplencia,
    RANK() OVER (ORDER BY taxa_inadimplencia DESC) AS ranking
FROM inadimplencia
ORDER BY ranking;
"""

print('Exercício 1 -- Ranking de inadimplência:')
run(sql_ex1)

Exercício 1 -- Ranking de inadimplência:
{'nome': 'Diego Rocha', 'total_parcelas': 0, 'parcelas_atrasadas': 0, 'taxa_inadimplencia': None, 'ranking': 1}
{'nome': 'Fabio Nunes', 'total_parcelas': 0, 'parcelas_atrasadas': 0, 'taxa_inadimplencia': None, 'ranking': 1}
{'nome': 'Bruno Souza', 'total_parcelas': 12, 'parcelas_atrasadas': 1, 'taxa_inadimplencia': Decimal('8.3'), 'ranking': 3}
{'nome': 'Gabriela Cruz', 'total_parcelas': 12, 'parcelas_atrasadas': 0, 'taxa_inadimplencia': Decimal('0.0'), 'ranking': 4}
{'nome': 'Isabela Teles', 'total_parcelas': 12, 'parcelas_atrasadas': 0, 'taxa_inadimplencia': Decimal('0.0'), 'ranking': 4}
{'nome': 'Ana Lima', 'total_parcelas': 24, 'parcelas_atrasadas': 0, 'taxa_inadimplencia': Decimal('0.0'), 'ranking': 4}
{'nome': 'Jonas Brito', 'total_parcelas': 12, 'parcelas_atrasadas': 0, 'taxa_inadimplencia': Decimal('0.0'), 'ranking': 4}
{'nome': 'Carla Mendes', 'total_parcelas': 24, 'parcelas_atrasadas': 0, 'taxa_inadimplencia': Decimal('0.0'), 'ranking'

[{'nome': 'Diego Rocha',
  'total_parcelas': 0,
  'parcelas_atrasadas': 0,
  'taxa_inadimplencia': None,
  'ranking': 1},
 {'nome': 'Fabio Nunes',
  'total_parcelas': 0,
  'parcelas_atrasadas': 0,
  'taxa_inadimplencia': None,
  'ranking': 1},
 {'nome': 'Bruno Souza',
  'total_parcelas': 12,
  'parcelas_atrasadas': 1,
  'taxa_inadimplencia': Decimal('8.3'),
  'ranking': 3},
 {'nome': 'Gabriela Cruz',
  'total_parcelas': 12,
  'parcelas_atrasadas': 0,
  'taxa_inadimplencia': Decimal('0.0'),
  'ranking': 4},
 {'nome': 'Isabela Teles',
  'total_parcelas': 12,
  'parcelas_atrasadas': 0,
  'taxa_inadimplencia': Decimal('0.0'),
  'ranking': 4},
 {'nome': 'Ana Lima',
  'total_parcelas': 24,
  'parcelas_atrasadas': 0,
  'taxa_inadimplencia': Decimal('0.0'),
  'ranking': 4},
 {'nome': 'Jonas Brito',
  'total_parcelas': 12,
  'parcelas_atrasadas': 0,
  'taxa_inadimplencia': Decimal('0.0'),
  'ranking': 4},
 {'nome': 'Carla Mendes',
  'total_parcelas': 24,
  'parcelas_atrasadas': 0,
  'taxa_inadi

In [26]:
# Exercício 2 -- Evolução do score médio mês a mês
# AVG OVER com ROWS BETWEEN calcula a média móvel acumulada (não a média do mês atual,
# mas a média de todos os meses até o atual). Útil para suavizar outliers mensais.

sql_ex2 = """
WITH score_mensal AS (
    SELECT
        DATE_TRUNC('month', created_at)::date AS mes,
        ROUND(AVG(score), 1)                  AS score_medio
    FROM propostas
    WHERE status = 'aprovada'
    GROUP BY 1
)
SELECT
    mes,
    score_medio,
    LAG(score_medio) OVER (ORDER BY mes)                          AS score_mes_anterior,
    ROUND(score_medio - LAG(score_medio) OVER (ORDER BY mes), 1)  AS variacao_score,
    ROUND(
        AVG(score_medio) OVER (
            ORDER BY mes
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
    , 1) AS media_movel_acumulada
FROM score_mensal
ORDER BY mes;
"""

print('Exercício 2 -- Evolução do score médio:')
run(sql_ex2)

Exercício 2 -- Evolução do score médio:
{'mes': datetime.date(2025, 10, 1), 'score_medio': Decimal('765.0'), 'score_mes_anterior': None, 'variacao_score': None, 'media_movel_acumulada': Decimal('765.0')}
{'mes': datetime.date(2025, 11, 1), 'score_medio': Decimal('900.0'), 'score_mes_anterior': Decimal('765.0'), 'variacao_score': Decimal('135.0'), 'media_movel_acumulada': Decimal('832.5')}
{'mes': datetime.date(2025, 12, 1), 'score_medio': Decimal('870.0'), 'score_mes_anterior': Decimal('900.0'), 'variacao_score': Decimal('-30.0'), 'media_movel_acumulada': Decimal('845.0')}
{'mes': datetime.date(2026, 1, 1), 'score_medio': Decimal('870.0'), 'score_mes_anterior': Decimal('870.0'), 'variacao_score': Decimal('0.0'), 'media_movel_acumulada': Decimal('851.3')}
{'mes': datetime.date(2026, 2, 1), 'score_medio': Decimal('735.0'), 'score_mes_anterior': Decimal('870.0'), 'variacao_score': Decimal('-135.0'), 'media_movel_acumulada': Decimal('828.0')}
{'mes': datetime.date(2026, 3, 1), 'score_medio

[{'mes': datetime.date(2025, 10, 1),
  'score_medio': Decimal('765.0'),
  'score_mes_anterior': None,
  'variacao_score': None,
  'media_movel_acumulada': Decimal('765.0')},
 {'mes': datetime.date(2025, 11, 1),
  'score_medio': Decimal('900.0'),
  'score_mes_anterior': Decimal('765.0'),
  'variacao_score': Decimal('135.0'),
  'media_movel_acumulada': Decimal('832.5')},
 {'mes': datetime.date(2025, 12, 1),
  'score_medio': Decimal('870.0'),
  'score_mes_anterior': Decimal('900.0'),
  'variacao_score': Decimal('-30.0'),
  'media_movel_acumulada': Decimal('845.0')},
 {'mes': datetime.date(2026, 1, 1),
  'score_medio': Decimal('870.0'),
  'score_mes_anterior': Decimal('870.0'),
  'variacao_score': Decimal('0.0'),
  'media_movel_acumulada': Decimal('851.3')},
 {'mes': datetime.date(2026, 2, 1),
  'score_medio': Decimal('735.0'),
  'score_mes_anterior': Decimal('870.0'),
  'variacao_score': Decimal('-135.0'),
  'media_movel_acumulada': Decimal('828.0')},
 {'mes': datetime.date(2026, 3, 1),
 

In [27]:
# Exercício 3 -- Top 3 propostas por cliente usando LATERAL
# ROW_NUMBER dentro do LATERAL atribui a posição. LEFT JOIN LATERAL preserva
# clientes sem propostas. LIMIT 3 dentro do subselect é o padrão clássico.

sql_ex3 = """
SELECT
    c.nome,
    top3.numero,
    top3.score,
    top3.status,
    top3.posicao
FROM clientes c
LEFT JOIN LATERAL (
    SELECT
        numero,
        score,
        status,
        ROW_NUMBER() OVER (ORDER BY score DESC) AS posicao
    FROM propostas p
    WHERE p.cliente_id = c.id
    ORDER BY score DESC
    LIMIT 3
) top3 ON true
ORDER BY c.nome, top3.posicao;
"""

print('Exercício 3 -- Top 3 propostas por score:')
run(sql_ex3)

Exercício 3 -- Top 3 propostas por score:
{'nome': 'Ana Lima', 'numero': 'PROP-012', 'score': 730, 'status': 'aprovada', 'posicao': 1}
{'nome': 'Ana Lima', 'numero': 'PROP-001', 'score': 720, 'status': 'aprovada', 'posicao': 2}
{'nome': 'Ana Lima', 'numero': 'PROP-002', 'score': 620, 'status': 'reprovada', 'posicao': 3}
{'nome': 'Bruno Souza', 'numero': 'PROP-013', 'score': 820, 'status': 'pendente', 'posicao': 1}
{'nome': 'Bruno Souza', 'numero': 'PROP-003', 'score': 810, 'status': 'aprovada', 'posicao': 2}
{'nome': 'Carla Mendes', 'numero': 'PROP-004', 'score': 900, 'status': 'aprovada', 'posicao': 1}
{'nome': 'Carla Mendes', 'numero': 'PROP-014', 'score': 880, 'status': 'aprovada', 'posicao': 2}
{'nome': 'Diego Rocha', 'numero': 'PROP-EMPATE-A', 'score': 900, 'status': 'aprovada', 'posicao': 1}
{'nome': 'Diego Rocha', 'numero': 'PROP-005', 'score': 410, 'status': 'reprovada', 'posicao': 2}
{'nome': 'Eva Santos', 'numero': 'PROP-006', 'score': 870, 'status': 'aprovada', 'posicao': 1}

[{'nome': 'Ana Lima',
  'numero': 'PROP-012',
  'score': 730,
  'status': 'aprovada',
  'posicao': 1},
 {'nome': 'Ana Lima',
  'numero': 'PROP-001',
  'score': 720,
  'status': 'aprovada',
  'posicao': 2},
 {'nome': 'Ana Lima',
  'numero': 'PROP-002',
  'score': 620,
  'status': 'reprovada',
  'posicao': 3},
 {'nome': 'Bruno Souza',
  'numero': 'PROP-013',
  'score': 820,
  'status': 'pendente',
  'posicao': 1},
 {'nome': 'Bruno Souza',
  'numero': 'PROP-003',
  'score': 810,
  'status': 'aprovada',
  'posicao': 2},
 {'nome': 'Carla Mendes',
  'numero': 'PROP-004',
  'score': 900,
  'status': 'aprovada',
  'posicao': 1},
 {'nome': 'Carla Mendes',
  'numero': 'PROP-014',
  'score': 880,
  'status': 'aprovada',
  'posicao': 2},
 {'nome': 'Diego Rocha',
  'numero': 'PROP-EMPATE-A',
  'score': 900,
  'status': 'aprovada',
  'posicao': 1},
 {'nome': 'Diego Rocha',
  'numero': 'PROP-005',
  'score': 410,
  'status': 'reprovada',
  'posicao': 2},
 {'nome': 'Eva Santos',
  'numero': 'PROP-006'

In [28]:
# Exercício 4 -- Distribuição de aprovações em quartis de score
# NTILE(4) em vez de NTILE(10) porque o seed tem 10 clientes.
# Com 10 clientes e NTILE(10), cada quartil teria exatamente 1 cliente -- trivial.
# NTILE(4) produz 2-3 clientes por grupo, o que já permite ver a distribuição.
# Para testar com NTILE(10), use o seed de 200k do notebook 01.

sql_ex4 = """
WITH score_cliente AS (
    SELECT
        c.id,
        c.nome,
        ROUND(AVG(p.score), 0) AS score_medio
    FROM clientes c
    JOIN propostas p ON p.cliente_id = c.id
    GROUP BY c.id, c.nome
),
quartis AS (
    SELECT
        *,
        NTILE(4) OVER (ORDER BY score_medio) AS quartil
    FROM score_cliente
)
SELECT
    quartil,
    MIN(score_medio) AS score_min,
    MAX(score_medio) AS score_max,
    COUNT(*)         AS qtd_clientes,
    -- Taxa de aprovação: propostas aprovadas / total de propostas dos clientes do quartil
    ROUND(
        (SELECT COUNT(*) FROM propostas p
         WHERE p.cliente_id IN (SELECT id FROM quartis q2 WHERE q2.quartil = quartis.quartil)
           AND p.status = 'aprovada')::numeric
        / NULLIF(
            (SELECT COUNT(*) FROM propostas p
             WHERE p.cliente_id IN (SELECT id FROM quartis q2 WHERE q2.quartil = quartis.quartil))
        , 0) * 100
    , 1) AS taxa_aprovacao_pct
FROM quartis
GROUP BY quartil
ORDER BY quartil;
"""

print('Exercício 4 -- Quartis de score:')
run(sql_ex4)

Exercício 4 -- Quartis de score:
{'quartil': 1, 'score_min': Decimal('530'), 'score_max': Decimal('690'), 'qtd_clientes': 3, 'taxa_aprovacao_pct': Decimal('50.0')}
{'quartil': 2, 'score_min': Decimal('740'), 'score_max': Decimal('790'), 'qtd_clientes': 3, 'taxa_aprovacao_pct': Decimal('75.0')}
{'quartil': 3, 'score_min': Decimal('815'), 'score_max': Decimal('870'), 'qtd_clientes': 2, 'taxa_aprovacao_pct': Decimal('66.7')}
{'quartil': 4, 'score_min': Decimal('890'), 'score_max': Decimal('950'), 'qtd_clientes': 2, 'taxa_aprovacao_pct': Decimal('100.0')}


[{'quartil': 1,
  'score_min': Decimal('530'),
  'score_max': Decimal('690'),
  'qtd_clientes': 3,
  'taxa_aprovacao_pct': Decimal('50.0')},
 {'quartil': 2,
  'score_min': Decimal('740'),
  'score_max': Decimal('790'),
  'qtd_clientes': 3,
  'taxa_aprovacao_pct': Decimal('75.0')},
 {'quartil': 3,
  'score_min': Decimal('815'),
  'score_max': Decimal('870'),
  'qtd_clientes': 2,
  'taxa_aprovacao_pct': Decimal('66.7')},
 {'quartil': 4,
  'score_min': Decimal('890'),
  'score_max': Decimal('950'),
  'qtd_clientes': 2,
  'taxa_aprovacao_pct': Decimal('100.0')}]

In [29]:
# Exercício 5 -- Primeira e última proposta por cliente
# Dois LEFT JOIN LATERAL: um com ORDER BY created_at ASC (primeira),
# outro com DESC (última). Ambos com LIMIT 1.
# Gotcha: se o cliente tem apenas uma proposta, primeira = última e variação = 0.

sql_ex5 = """
SELECT
    c.nome,
    c.renda_mensal,
    primeira.numero     AS primeira_proposta,
    primeira.valor      AS primeiro_valor,
    primeira.status     AS primeiro_status,
    ultima.numero       AS ultima_proposta,
    ultima.valor        AS ultimo_valor,
    ultima.status       AS ultimo_status,
    COALESCE(ultima.valor - primeira.valor, 0) AS variacao_valor
FROM clientes c
LEFT JOIN LATERAL (
    SELECT numero, valor, status, created_at
    FROM propostas p
    WHERE p.cliente_id = c.id
    ORDER BY p.created_at ASC
    LIMIT 1
) primeira ON true
LEFT JOIN LATERAL (
    SELECT numero, valor, status, created_at
    FROM propostas p
    WHERE p.cliente_id = c.id
    ORDER BY p.created_at DESC
    LIMIT 1
) ultima ON true
ORDER BY c.nome;
"""

print('Exercício 5 -- Primeira e última proposta por cliente:')
run(sql_ex5)

Exercício 5 -- Primeira e última proposta por cliente:
{'nome': 'Ana Lima', 'renda_mensal': Decimal('3500.00'), 'primeira_proposta': 'PROP-001', 'primeiro_valor': Decimal('5000.00'), 'primeiro_status': 'aprovada', 'ultima_proposta': 'PROP-012', 'ultimo_valor': Decimal('7000.00'), 'ultimo_status': 'aprovada', 'variacao_valor': Decimal('2000.00')}
{'nome': 'Bruno Souza', 'renda_mensal': Decimal('8200.00'), 'primeira_proposta': 'PROP-003', 'primeiro_valor': Decimal('30000.00'), 'primeiro_status': 'aprovada', 'ultima_proposta': 'PROP-013', 'ultimo_valor': Decimal('20000.00'), 'ultimo_status': 'pendente', 'variacao_valor': Decimal('-10000.00')}
{'nome': 'Carla Mendes', 'renda_mensal': Decimal('15000.00'), 'primeira_proposta': 'PROP-004', 'primeiro_valor': Decimal('80000.00'), 'primeiro_status': 'aprovada', 'ultima_proposta': 'PROP-014', 'ultimo_valor': Decimal('60000.00'), 'ultimo_status': 'aprovada', 'variacao_valor': Decimal('-20000.00')}
{'nome': 'Diego Rocha', 'renda_mensal': Decimal('2

[{'nome': 'Ana Lima',
  'renda_mensal': Decimal('3500.00'),
  'primeira_proposta': 'PROP-001',
  'primeiro_valor': Decimal('5000.00'),
  'primeiro_status': 'aprovada',
  'ultima_proposta': 'PROP-012',
  'ultimo_valor': Decimal('7000.00'),
  'ultimo_status': 'aprovada',
  'variacao_valor': Decimal('2000.00')},
 {'nome': 'Bruno Souza',
  'renda_mensal': Decimal('8200.00'),
  'primeira_proposta': 'PROP-003',
  'primeiro_valor': Decimal('30000.00'),
  'primeiro_status': 'aprovada',
  'ultima_proposta': 'PROP-013',
  'ultimo_valor': Decimal('20000.00'),
  'ultimo_status': 'pendente',
  'variacao_valor': Decimal('-10000.00')},
 {'nome': 'Carla Mendes',
  'renda_mensal': Decimal('15000.00'),
  'primeira_proposta': 'PROP-004',
  'primeiro_valor': Decimal('80000.00'),
  'primeiro_status': 'aprovada',
  'ultima_proposta': 'PROP-014',
  'ultimo_valor': Decimal('60000.00'),
  'ultimo_status': 'aprovada',
  'variacao_valor': Decimal('-20000.00')},
 {'nome': 'Diego Rocha',
  'renda_mensal': Decimal(